In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# TASK1_DATA_DIR contains the train and test folders.
DATA_DIR = Path(os.environ.get("TASK1_DATA_DIR", "data"))
SAMPLE_SUBMISSION = Path(os.environ.get(
    "TASK1_SAMPLE_SUBMISSION", str(DATA_DIR / "submission-for-task2.csv")
))
ARTIFACT_DIR = Path(os.environ.get("TASK1_OUTPUT_DIR", "outputs"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 确认根地址
import os
import pandas as pd

DATA_ROOT = str(DATA_DIR)
train_dir = os.path.join(DATA_ROOT, "train")
test_dir = os.path.join(DATA_ROOT, "test")
submission_path = str(SAMPLE_SUBMISSION)

print("===== TRAIN =====")
print(os.listdir(train_dir)[:30])

print("\n===== TEST =====")
print(os.listdir(test_dir)[:30])

print("\n===== SUBMISSION =====")
submission = pd.read_csv(submission_path)

print(submission.head())
print("\ncolumns:", submission.columns.tolist())
print("shape:", submission.shape)

In [ ]:
import os

for root, dirs, files in os.walk(str(DATA_DIR)):
    if "train" in dirs:
        print("找到 train：", os.path.join(root, "train"))
    
    if "test" in dirs:
        print("找到 test：", os.path.join(root, "test"))
    
    if "submission-for-task2.csv" in files:
        print(
            "找到 submission：",
            os.path.join(root, "submission-for-task2.csv")
        )

In [ ]:
#检查图片数量
print("训练集内容：")

for item in os.listdir(train_dir)[:20]:
    path = os.path.join(train_dir, item)

    if os.path.isdir(path):
        print(
            "类别:",
            item,
            "| 图片数量:",
            len(os.listdir(path))
        )

In [ ]:
# 统计训练集类别和图片数
import os
import pandas as pd

classes = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

print("类别数量：", len(classes))
print("\n类别名称：")

for i, cls in enumerate(classes):
    print(i, cls)

class_counts = {}

for cls in classes:
    cls_path = os.path.join(train_dir, cls)

    images = [
        f for f in os.listdir(cls_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    class_counts[cls] = len(images)

count_df = pd.DataFrame({
    "class": list(class_counts.keys()),
    "count": list(class_counts.values())
})

count_df = count_df.sort_values("count", ascending=False)

print(count_df)

print("\n总训练图片数量：", count_df["count"].sum())
print("最多类别图片数：", count_df["count"].max())
print("最少类别图片数：", count_df["count"].min())

In [ ]:
# 绘制对应类别图示
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.bar(
    count_df["class"],
    count_df["count"]
)

plt.xticks(
    rotation=60,
    ha="right"
)

plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("Training Set Class Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# 随机检查对应类别植物
import random
from PIL import Image
import matplotlib.pyplot as plt

num_classes_to_show = min(len(classes), 12)

selected_classes = random.sample(
    classes,
    num_classes_to_show
)

plt.figure(figsize=(15, 10))

for i, cls in enumerate(selected_classes):

    cls_path = os.path.join(train_dir, cls)

    images = [
        f for f in os.listdir(cls_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    img_name = random.choice(images)

    img_path = os.path.join(
        cls_path,
        img_name
    )

    img = Image.open(img_path).convert("RGB")

    plt.subplot(3, 4, i + 1)

    plt.imshow(img)

    plt.title(cls)

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 检查图片尺寸确认合适的模型输入
from collections import Counter
from PIL import Image

sizes = []

for cls in classes:

    cls_path = os.path.join(train_dir, cls)

    images = os.listdir(cls_path)[:50]

    for img_name in images:

        img_path = os.path.join(
            cls_path,
            img_name
        )

        try:
            img = Image.open(img_path)
            sizes.append(img.size)

        except:
            pass

size_counter = Counter(sizes)

print("最常见的图片尺寸：")

for size, count in size_counter.most_common(10):
    print(size, count)

In [ ]:
# 检查submission
submission = pd.read_csv(submission_path)

print(submission.head(10))

print("\n列名：")
print(submission.columns.tolist())

print("\nshape：")
print(submission.shape)

print("\n数据类型：")
print(submission.dtypes)

In [ ]:
# 先验证 test 和 submission 是否完全对应
import os
import pandas as pd

submission = pd.read_csv(submission_path)

test_files = [
    f for f in os.listdir(test_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

print("test 图片数量:", len(test_files))
print("submission 行数:", len(submission))

test_set = set(test_files)
sub_set = set(submission["ID"])

print("submission 中找不到对应图片:", len(sub_set - test_set))
print("test 中没有出现在 submission:", len(test_set - sub_set))

In [ ]:
# 建立 Stratified 训练集 / 验证集
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

records = []

for class_name in classes:
    class_dir = os.path.join(train_dir, class_name)

    for filename in os.listdir(class_dir):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            records.append({
                "path": os.path.join(class_dir, filename),
                "label": class_name
            })

df = pd.DataFrame(records)

print(df.head())
print("\n总样本数:", len(df))

# 分层划分
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

print("\nTrain 类别：")
print(train_df["label"].value_counts())

print("\nValidation 类别：")
print(val_df["label"].value_counts())

In [ ]:
# 建立类别编号
class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

idx_to_class = {
    idx: cls
    for cls, idx in class_to_idx.items()
}

print(class_to_idx)

In [ ]:
# 设计图像预处理,实现数据增强
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 验证集则绝对不能随机增强
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# 构造dataset
from torch.utils.data import Dataset
from PIL import Image


class PlantDataset(Dataset):

    def __init__(
        self,
        dataframe,
        class_to_idx,
        transform=None
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        label = self.class_to_idx[
            row["label"]
        ]

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = PlantDataset(
    train_df,
    class_to_idx,
    transform=train_transform
)

val_dataset = PlantDataset(
    val_df,
    class_to_idx,
    transform=val_transform
)

print("Train Dataset:", len(train_dataset))
print("Validation Dataset:", len(val_dataset))

In [ ]:
# dataloader 并检查
import torch

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("labels:", labels[:10])

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("device:", device)

In [ ]:
# 加载预训练模型
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

In [ ]:
# 原 ResNet18 最后一层：512 → 1000   因为我们只有 5 类，所以改成：512 → 5
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(classes)
)

model = model.to(device)

print(model.fc)

In [ ]:
# 设置Loss 和优化器 并且引入F1 
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# 引入F1
from sklearn.metrics import f1_score, accuracy_score
# 建立验证函数
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * images.size(0)
            )

            preds = outputs.argmax(dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return loss, accuracy, macro_f1



In [ ]:
# 第一版先训练小样本
EPOCHS = 15

best_f1 = 0.0
best_model_path = str(ARTIFACT_DIR / "best_resnet18.pth")

history = []

for epoch in range(EPOCHS):

    # =========================
    # Train
    # =========================

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    train_loss = (
        running_loss /
        len(train_loader.dataset)
    )

    # =========================
    # Validation
    # =========================

    val_loss, val_acc, val_f1 = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_f1)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # 保存 F1 最高的模型
    if val_f1 > best_f1:

        best_f1 = val_f1

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print(
            f"  -> Best model saved "
            f"(F1={best_f1:.4f})"
        )

In [ ]:
# 先找出模型到底错在哪些类别
model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model.eval()

print("Best model loaded.")

In [ ]:
# 进行相关数据计算
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
# 输出
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=classes,
        digits=4
    )
)

In [ ]:
# 检查混淆矩阵
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(
    all_labels,
    all_preds
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

fig, ax = plt.subplots(
    figsize=(9, 7)
)

disp.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title(
    "ResNet18 Validation Confusion Matrix"
)

plt.tight_layout()
plt.show()

In [ ]:
# 修改数据增强能力
from torchvision import transforms

train_transform_v2 = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(25),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# 重新构造 dataset
train_dataset_v2 = PlantDataset(
    train_df,
    class_to_idx,
    transform=train_transform_v2
)

val_dataset_v2 = PlantDataset(
    val_df,
    class_to_idx,
    transform=val_transform
)

train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_v2 = DataLoader(
    val_dataset_v2,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# 使用两阶段学习

# 冻结当前数据
import torch
import torch.nn as nn
from torchvision import models

model_v2 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# 冻结所有预训练参数
for param in model_v2.parameters():
    param.requires_grad = False

# 替换分类器
num_features = model_v2.fc.in_features

model_v2.fc = nn.Linear(
    num_features,
    5
)

model_v2 = model_v2.to(device)

# 加入顺滑器
criterion_v2 = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

optimizer_v2 = torch.optim.AdamW(
    model_v2.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
# 跑3样本测试
for epoch in range(3):

    model_v2.train()

    running_loss = 0.0

    for images, labels in train_loader_v2:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_v2.zero_grad()

        outputs = model_v2(images)

        loss = criterion_v2(
            outputs,
            labels
        )

        loss.backward()
        optimizer_v2.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    train_loss = (
        running_loss /
        len(train_loader_v2.dataset)
    )

    val_loss, val_acc, val_f1 = evaluate(
        model_v2,
        val_loader_v2,
        criterion_v2,
        device
    )

    print(
        f"Stage1 Epoch [{epoch+1}/3] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

In [ ]:
# 解冻进行 Fine-tuning
for param in model_v2.parameters():
    param.requires_grad = True

optimizer_v2 = torch.optim.AdamW(
    model_v2.parameters(),
    lr=5e-5,
    weight_decay=5e-4
)

scheduler_v2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v2,
    mode="max",
    factor=0.5,
    patience=2
)

# 在训练
best_f1_v2 = 0.0

best_model_path_v2 = (
    str(ARTIFACT_DIR / "best_resnet18_v2.pth")
)

history_v2 = []

for epoch in range(12):

    model_v2.train()

    running_loss = 0.0

    for images, labels in train_loader_v2:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_v2.zero_grad()

        outputs = model_v2(images)

        loss = criterion_v2(
            outputs,
            labels
        )

        loss.backward()

        optimizer_v2.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    train_loss = (
        running_loss /
        len(train_loader_v2.dataset)
    )

    val_loss, val_acc, val_f1 = evaluate(
        model_v2,
        val_loader_v2,
        criterion_v2,
        device
    )

    scheduler_v2.step(val_f1)

    history_v2.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(
        f"Stage2 Epoch [{epoch+1:02d}/12] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1_v2:

        best_f1_v2 = val_f1

        torch.save(
            model_v2.state_dict(),
            best_model_path_v2
        )

        print(
            f" -> Best V2 saved "
            f"(F1={best_f1_v2:.4f})"
        )

In [ ]:
from torchvision import transforms

train_transform_v3 = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# 重构 dataloader
train_dataset_v3 = PlantDataset(
    train_df,
    class_to_idx,
    transform=train_transform_v3
)

val_dataset_v3 = PlantDataset(
    val_df,
    class_to_idx,
    transform=val_transform
)

train_loader_v3 = DataLoader(
    train_dataset_v3,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_v3 = DataLoader(
    val_dataset_v3,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 恢复模型
import torch
import torch.nn as nn
from torchvision import models

model_v3 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

num_features = model_v3.fc.in_features

model_v3.fc = nn.Linear(
    num_features,
    5
)

model_v3 = model_v3.to(device)
criterion_v3 = nn.CrossEntropyLoss()
optimizer_v3 = torch.optim.AdamW(
    model_v3.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler_v3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v3,
    mode="max",
    factor=0.5,
    patience=2
)
EPOCHS = 15

best_f1_v3 = 0.0
best_model_path_v3 = str(ARTIFACT_DIR / "best_resnet18_v3.pth")

history_v3 = []

for epoch in range(EPOCHS):

    model_v3.train()

    running_loss = 0.0

    for images, labels in train_loader_v3:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_v3.zero_grad()

        outputs = model_v3(images)

        loss = criterion_v3(
            outputs,
            labels
        )

        loss.backward()

        optimizer_v3.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    train_loss = (
        running_loss /
        len(train_loader_v3.dataset)
    )

    val_loss, val_acc, val_f1 = evaluate(
        model_v3,
        val_loader_v3,
        criterion_v3,
        device
    )

    scheduler_v3.step(val_f1)

    history_v3.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1_v3:

        best_f1_v3 = val_f1

        torch.save(
            model_v3.state_dict(),
            best_model_path_v3
        )

        print(
            f" -> Best V3 saved "
            f"(F1={best_f1_v3:.4f})"
        )

In [ ]:
from torchvision import transforms

train_transform_v4 = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset_v4 = PlantDataset(
    train_df,
    class_to_idx,
    transform=train_transform_v4
)

val_dataset_v4 = PlantDataset(
    val_df,
    class_to_idx,
    transform=val_transform
)

train_loader_v4 = DataLoader(
    train_dataset_v4,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_v4 = DataLoader(
    val_dataset_v4,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
import torch
import torch.nn as nn
from torchvision import models

model_v4 = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)
num_features = model_v4.classifier[1].in_features

model_v4.classifier[1] = nn.Linear(
    num_features,
    5
)

model_v4 = model_v4.to(device)

print(model_v4.classifier)

criterion_v4 = nn.CrossEntropyLoss()
optimizer_v4 = torch.optim.AdamW(
    model_v4.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler_v4 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v4,
    mode="max",
    factor=0.5,
    patience=2
)

EPOCHS = 15

best_f1_v4 = 0.0

best_model_path_v4 = (
    str(ARTIFACT_DIR / "best_efficientnet_b0.pth")
)

history_v4 = []

for epoch in range(EPOCHS):

    model_v4.train()

    running_loss = 0.0

    for images, labels in train_loader_v4:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_v4.zero_grad()

        outputs = model_v4(images)

        loss = criterion_v4(
            outputs,
            labels
        )

        loss.backward()

        optimizer_v4.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    train_loss = (
        running_loss /
        len(train_loader_v4.dataset)
    )

    val_loss, val_acc, val_f1 = evaluate(
        model_v4,
        val_loader_v4,
        criterion_v4,
        device
    )

    scheduler_v4.step(val_f1)

    history_v4.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1_v4:

        best_f1_v4 = val_f1

        torch.save(
            model_v4.state_dict(),
            best_model_path_v4
        )

        print(
            f" -> Best EfficientNet saved "
            f"(F1={best_f1_v4:.4f})"
        )

In [ ]:
model_v4.load_state_dict(
    torch.load(
        best_model_path_v4,
        map_location=device
    )
)

model_v4.eval()
import numpy as np

all_preds_v4 = []
all_labels_v4 = []

with torch.no_grad():

    for images, labels in val_loader_v4:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_v4(images)

        preds = outputs.argmax(dim=1)

        all_preds_v4.extend(
            preds.cpu().numpy()
        )

        all_labels_v4.extend(
            labels.cpu().numpy()
        )

all_preds_v4 = np.array(all_preds_v4)
all_labels_v4 = np.array(all_labels_v4)
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels_v4,
        all_preds_v4,
        target_names=classes,
        digits=4
    )
)
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

cm_v4 = confusion_matrix(
    all_labels_v4,
    all_preds_v4
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_v4,
    display_labels=classes
)

fig, ax = plt.subplots(figsize=(9, 7))

disp.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title(
    "EfficientNet-B0 Validation Confusion Matrix"
)

plt.tight_layout()
plt.show()


In [ ]:
# 5-fold划分
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("总样本数:", len(df))
print(df["label"].value_counts())

In [ ]:
# 建立模型函数
import torch
import torch.nn as nn
from torchvision import models


def create_model(model_name, num_classes=5):

    if model_name == "resnet18":

        model = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        num_features = model.fc.in_features

        model.fc = nn.Linear(
            num_features,
            num_classes
        )

    elif model_name == "efficientnet_b0":

        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        num_features = model.classifier[1].in_features

        model.classifier[1] = nn.Linear(
            num_features,
            num_classes
        )

    else:

        raise ValueError(
            f"Unknown model name: {model_name}"
        )

    return model.to(device)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import torch


def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item()
                * images.size(0)
            )

            preds = outputs.argmax(
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    val_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    val_acc = accuracy_score(
        all_labels,
        all_preds
    )

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return (
        val_loss,
        val_acc,
        val_f1
    )

In [ ]:
import os
import torch
import torch.nn as nn

from torch.utils.data import DataLoader


def train_one_fold(
    model_name,
    fold,
    train_fold_df,
    val_fold_df,
    epochs=15
):
    """
    训练单个 fold，并保存该 fold 上 Macro F1 最好的模型。

    参数：
    model_name:
        "resnet18"
        或
        "efficientnet_b0"

    fold:
        当前 fold 编号，1~5

    train_fold_df:
        当前 fold 的训练 DataFrame

    val_fold_df:
        当前 fold 的验证 DataFrame

    epochs:
        训练轮数

    返回：
        best_f1
        best_epoch
        best_path
    """

    print("\n" + "=" * 60)
    print(f"{model_name} | Fold {fold}")
    print("=" * 60)

    # =========================================================
    # 1. Dataset
    # =========================================================

    train_dataset_fold = PlantDataset(
        train_fold_df,
        class_to_idx,
        transform=train_transform
    )

    val_dataset_fold = PlantDataset(
        val_fold_df,
        class_to_idx,
        transform=val_transform
    )

    print(
        f"Train samples: {len(train_dataset_fold)} | "
        f"Val samples: {len(val_dataset_fold)}"
    )

    # =========================================================
    # 2. DataLoader
    # =========================================================

    train_loader_fold = DataLoader(
        train_dataset_fold,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader_fold = DataLoader(
        val_dataset_fold,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # =========================================================
    # 3. 创建全新的模型
    # =========================================================

    model = create_model(
        model_name,
        num_classes=len(classes)
    )

    # =========================================================
    # 4. Loss / Optimizer / Scheduler
    # =========================================================

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    # =========================================================
    # 5. 保存路径
    # =========================================================

    best_path = (
        str(ARTIFACT_DIR / f"{model_name}_fold{fold}.pth")
    )

    best_f1 = -1.0
    best_epoch = 0
    best_acc = 0.0

    # =========================================================
    # 6. Train
    # =========================================================

    for epoch in range(epochs):

        # -------------------------
        # Train
        # -------------------------

        model.train()

        running_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader_fold:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            # loss
            running_loss += (
                loss.item()
                * images.size(0)
            )

            # train accuracy
            preds = outputs.argmax(
                dim=1
            )

            train_correct += (
                preds == labels
            ).sum().item()

            train_total += (
                labels.size(0)
            )

        train_loss = (
            running_loss
            /
            len(train_loader_fold.dataset)
        )

        train_acc = (
            train_correct
            /
            train_total
        )

        # -------------------------
        # Validation
        # -------------------------

        val_loss, val_acc, val_f1 = evaluate(
            model,
            val_loader_fold,
            criterion,
            device
        )

        # 根据 Macro F1 调整学习率
        scheduler.step(val_f1)

        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        print(
            f"Fold {fold} | "
            f"Epoch [{epoch+1:02d}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val F1: {val_f1:.4f} | "
            f"LR: {current_lr:.2e}"
        )

        # =====================================================
        # 7. 保存该 Fold 最佳模型
        # =====================================================

        if val_f1 > best_f1:

            best_f1 = val_f1
            best_epoch = epoch + 1
            best_acc = val_acc

            torch.save(
                model.state_dict(),
                best_path
            )

            print(
                f"  -> Best model saved | "
                f"Epoch={best_epoch} | "
                f"F1={best_f1:.4f} | "
                f"Path={best_path}"
            )

    # =========================================================
    # 8. 当前 Fold 完成
    # =========================================================

    print("\n" + "-" * 60)

    print(
        f"{model_name} Fold {fold} BEST\n"
        f"Best Epoch : {best_epoch}\n"
        f"Best Acc   : {best_acc:.4f}\n"
        f"Best F1    : {best_f1:.4f}\n"
        f"Saved Path : {best_path}"
    )

    print("-" * 60)

    # 清一下显存
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_f1, best_epoch, best_path

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resnet_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(
        df,
        df["label"]
    ),
    start=1
):

    train_fold_df = (
        df.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_fold_df = (
        df.iloc[val_idx]
        .reset_index(drop=True)
    )

    best_f1, best_epoch, best_path = train_one_fold(
        model_name="resnet18",
        fold=fold,
        train_fold_df=train_fold_df,
        val_fold_df=val_fold_df,
        epochs=15
    )

    resnet_results.append({
        "fold": fold,
        "f1": best_f1,
        "epoch": best_epoch,
        "path": best_path
    })

In [ ]:
# 统计
resnet_f1 = [
    x["f1"]
    for x in resnet_results
]

print("\n===== ResNet18 5-Fold =====")

for r in resnet_results:
    print(
        f"Fold {r['fold']}: "
        f"F1={r['f1']:.4f}, "
        f"Best Epoch={r['epoch']}"
    )

print(
    "\nMean F1:",
    np.mean(resnet_f1)
)

print(
    "Std F1:",
    np.std(resnet_f1)
)

In [ ]:
skf_eff = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

efficientnet_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf_eff.split(
        df,
        df["label"]
    ),
    start=1
):

    train_fold_df = (
        df.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_fold_df = (
        df.iloc[val_idx]
        .reset_index(drop=True)
    )

    best_f1, best_epoch, best_path = train_one_fold(
        model_name="efficientnet_b0",
        fold=fold,
        train_fold_df=train_fold_df,
        val_fold_df=val_fold_df,
        epochs=15
    )

    efficientnet_results.append({
        "fold": fold,
        "f1": best_f1,
        "epoch": best_epoch,
        "path": best_path
    })

In [ ]:
efficientnet_f1 = [
    x["f1"]
    for x in efficientnet_results
]

print(
    "\n===== EfficientNet-B0 5-Fold ====="
)

for r in efficientnet_results:

    print(
        f"Fold {r['fold']}: "
        f"F1={r['f1']:.4f}, "
        f"Best Epoch={r['epoch']}"
    )

print(
    "\nMean F1:",
    np.mean(efficientnet_f1)
)

print(
    "Std F1:",
    np.std(efficientnet_f1)
)

In [ ]:
print("\n===== Saved Fold Models =====")

for filename in sorted(
    os.listdir(str(ARTIFACT_DIR))
):
    if (
        "fold" in filename
        and filename.endswith(".pth")
    ):
        print(filename)

In [ ]:
# 建立训练集
FINAL_EPOCHS = 13
print(len(df))
print(df["label"].value_counts())

final_train_dataset = PlantDataset(
    df,
    class_to_idx,
    transform=train_transform
)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# 初始化全新模型
from torchvision import models
import torch.nn as nn

final_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

num_features = final_model.fc.in_features

final_model.fc = nn.Linear(
    num_features,
    5
)

final_model = final_model.to(device)

final_criterion = nn.CrossEntropyLoss()

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    final_optimizer,
    T_max=13
)



In [ ]:
# 全量训练
FINAL_EPOCHS = 13

for epoch in range(FINAL_EPOCHS):

    final_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in final_train_loader:

        images = images.to(device)
        labels = labels.to(device)

        final_optimizer.zero_grad()

        outputs = final_model(images)

        loss = final_criterion(
            outputs,
            labels
        )

        loss.backward()

        final_optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        preds = outputs.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    final_scheduler.step()

    train_loss = (
        running_loss /
        len(final_train_loader.dataset)
    )

    train_acc = (
        correct /
        total
    )

    print(
        f"Epoch [{epoch+1:02d}/{FINAL_EPOCHS}] "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f}"
    )



In [ ]:
FINAL_MODEL_PATH = (
    str(ARTIFACT_DIR / "final_resnet18.pth")
)

torch.save(
    final_model.state_dict(),
    FINAL_MODEL_PATH
)

print("Final model saved.")

In [ ]:
# 构建test dataset 
from torch.utils.data import Dataset
from PIL import Image
import os


class PlantTestDataset(Dataset):

    def __init__(
        self,
        test_dir,
        filenames,
        transform=None
    ):

        self.test_dir = test_dir
        self.filenames = filenames
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):

        filename = self.filenames[idx]

        image_path = os.path.join(
            self.test_dir,
            filename
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, filename

In [ ]:
submission = pd.read_csv(
    submission_path
)

test_filenames = (
    submission["ID"]
    .tolist()
)
test_dataset = PlantTestDataset(
    test_dir,
    test_filenames,
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print(
    "Test images:",
    len(test_dataset)
)

print(
    "Submission rows:",
    len(submission)
)

In [ ]:
# 进行预测
final_model.eval()

test_predictions = []

with torch.no_grad():

    for images, filenames in test_loader:

        images = images.to(device)

        outputs = final_model(images)

        preds = outputs.argmax(
            dim=1
        )

        test_predictions.extend(
            preds.cpu().numpy()
        )

print(len(test_predictions))

In [ ]:
# 转换回植物名称
predicted_classes = [
    idx_to_class[int(idx)]
    for idx in test_predictions
]

print(predicted_classes[:20])



In [ ]:
# 生成 submission
final_submission = submission.copy()

final_submission["Category"] = (
    predicted_classes
)

OUTPUT_PATH = (
    str(ARTIFACT_DIR / "submission.csv")
)

final_submission.to_csv(
    OUTPUT_PATH,
    index=False
)

print(final_submission.head(10))

print("\nSaved to:")
print(OUTPUT_PATH)

print(
    final_submission.shape
)

print(
    final_submission.columns.tolist()
)

print(
    final_submission[
        "Category"
    ].value_counts()
)



In [ ]:
# 最后检查
print(
    final_submission.shape
)
print(
    final_submission.columns.tolist()
)
print(
    final_submission[
        "Category"
    ].value_counts()
)


In [ ]:
# ============================================================
# 植物分类最终 Ensemble + TTA 一键推理脚本
#
# 功能：
# 1. 自动定义所有路径
# 2. 自动读取 submission 模板
# 3. 自动读取 test 图片
# 4. 加载 5 个 ResNet18 Fold
# 5. 加载 5 个 EfficientNet-B0 Fold
# 6. 每个模型进行 4-view TTA
# 7. Fold 内 Soft Voting
# 8. 两种模型 0.5 / 0.5 Soft Voting
# 9. 自动生成 submission_ensemble_tta.csv
#
# 不依赖之前 Notebook 中定义的变量
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

import os
import gc

import pandas as pd
import numpy as np

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


# ============================================================
# 1. 固定路径
# ============================================================

DATA_ROOT = str(DATA_DIR)


TEST_DIR = os.path.join(
    DATA_ROOT,
    "test"
)

SUBMISSION_PATH = str(SAMPLE_SUBMISSION)

MODEL_DIR = str(ARTIFACT_DIR)

OUTPUT_PATH = (
    str(ARTIFACT_DIR / "submission_ensemble_tta.csv")
)


# ============================================================
# 2. 类别定义
#    一定与训练时完全相同
# ============================================================

classes = [
    "Black-grass",
    "Common wheat",
    "Loose Silky-bent",
    "Scentless Mayweed",
    "Sugar beet"
]

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

idx_to_class = {
    idx: cls
    for cls, idx in class_to_idx.items()
}

NUM_CLASSES = len(classes)


# ============================================================
# 3. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 60)
print("Device:", device)
print("=" * 60)


# ============================================================
# 4. 检查必要文件
# ============================================================

if not os.path.exists(SUBMISSION_PATH):
    raise FileNotFoundError(
        f"找不到 submission 模板：\n{SUBMISSION_PATH}"
    )

if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(
        f"找不到 test 目录：\n{TEST_DIR}"
    )


# 检查10个模型
expected_model_files = []

for fold in range(1, 6):

    expected_model_files.append(
        f"resnet18_fold{fold}.pth"
    )

    expected_model_files.append(
        f"efficientnet_b0_fold{fold}.pth"
    )


missing_models = []

for filename in expected_model_files:

    path = os.path.join(
        MODEL_DIR,
        filename
    )

    if not os.path.exists(path):
        missing_models.append(filename)


if missing_models:

    print("\n缺少以下模型：")

    for f in missing_models:
        print(" -", f)

    raise FileNotFoundError(
        "10个 Fold 模型不完整，请检查 TASK1_OUTPUT_DIR"
    )

print("10 个 Fold 模型全部存在。")


# ============================================================
# 5. 读取 Submission 模板
# ============================================================

submission = pd.read_csv(
    SUBMISSION_PATH
)

print("\nSubmission shape:")
print(submission.shape)

print("\nSubmission columns:")
print(submission.columns.tolist())

print("\nSubmission head:")
print(submission.head())


if "ID" not in submission.columns:
    raise ValueError(
        "Submission 中没有 ID 列。"
    )

if "Category" not in submission.columns:
    raise ValueError(
        "Submission 中没有 Category 列。"
    )


test_filenames = (
    submission["ID"]
    .astype(str)
    .tolist()
)

print(
    "\n测试样本数量:",
    len(test_filenames)
)


# ============================================================
# 6. 自动扫描 test 目录
#
# 即使 test 内部还有文件夹，也可以正常找到
# ============================================================

test_file_map = {}

for root, dirs, files in os.walk(TEST_DIR):

    for filename in files:

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):
            test_file_map[filename] = os.path.join(
                root,
                filename
            )


print(
    "test 目录实际扫描到图片:",
    len(test_file_map)
)


missing_test_images = [
    filename
    for filename in test_filenames
    if filename not in test_file_map
]

if missing_test_images:

    print(
        "\nSubmission 中有图片 "
        "在 test 目录找不到：",
        len(missing_test_images)
    )

    print(
        "示例:",
        missing_test_images[:10]
    )

    raise FileNotFoundError(
        "Test 图片与 Submission ID 无法完全对应。"
    )

print(
    "Submission ID 与 Test 图片对应正常。"
)


# ============================================================
# 7. Test Transform
#
# 与之前 Validation Transform 保持一致
# ============================================================

test_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 8. Test Dataset
# ============================================================

class PlantTestDataset(Dataset):

    def __init__(
        self,
        filenames,
        file_map,
        transform=None
    ):

        self.filenames = filenames
        self.file_map = file_map
        self.transform = transform

    def __len__(self):

        return len(
            self.filenames
        )

    def __getitem__(
        self,
        idx
    ):

        filename = (
            self.filenames[idx]
        )

        image_path = (
            self.file_map[
                filename
            ]
        )

        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )

        if self.transform:

            image = (
                self.transform(
                    image
                )
            )

        return (
            image,
            filename
        )


test_dataset = PlantTestDataset(
    filenames=test_filenames,
    file_map=test_file_map,
    transform=test_transform
)


test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print(
    "\nTest Dataset:",
    len(test_dataset)
)


# ============================================================
# 9. 创建模型函数
#
# 注意：
# 这里 weights=None
# 因为我们马上加载自己的 checkpoint，
# 不需要重新下载 ImageNet 权重。
# ============================================================

def create_model(
    model_name,
    num_classes=5
):

    if model_name == "resnet18":

        model = models.resnet18(
            weights=None
        )

        num_features = (
            model.fc.in_features
        )

        model.fc = nn.Linear(
            num_features,
            num_classes
        )


    elif model_name == "efficientnet_b0":

        model = models.efficientnet_b0(
            weights=None
        )

        num_features = (
            model
            .classifier[1]
            .in_features
        )

        model.classifier[1] = (
            nn.Linear(
                num_features,
                num_classes
            )
        )


    else:

        raise ValueError(
            f"未知模型: {model_name}"
        )


    return model.to(
        device
    )


# ============================================================
# 10. 安全加载 checkpoint
# ============================================================

def load_checkpoint(
    model,
    model_path
):

    try:

        state_dict = torch.load(
            model_path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        # 兼容较老 PyTorch
        state_dict = torch.load(
            model_path,
            map_location=device
        )

    model.load_state_dict(
        state_dict
    )

    return model


# ============================================================
# 11. TTA
#
# 4种：
# Original
# Horizontal Flip
# Vertical Flip
# Horizontal + Vertical
# ============================================================

def predict_tta(
    model,
    images
):

    probs_list = []


    # --------------------------
    # 1. Original
    # --------------------------

    logits = model(
        images
    )

    probs = F.softmax(
        logits,
        dim=1
    )

    probs_list.append(
        probs
    )


    # --------------------------
    # 2. Horizontal Flip
    # --------------------------

    images_h = torch.flip(
        images,
        dims=[3]
    )

    logits_h = model(
        images_h
    )

    probs_h = F.softmax(
        logits_h,
        dim=1
    )

    probs_list.append(
        probs_h
    )


    # --------------------------
    # 3. Vertical Flip
    # --------------------------

    images_v = torch.flip(
        images,
        dims=[2]
    )

    logits_v = model(
        images_v
    )

    probs_v = F.softmax(
        logits_v,
        dim=1
    )

    probs_list.append(
        probs_v
    )


    # --------------------------
    # 4. H + V
    # --------------------------

    images_hv = torch.flip(
        images,
        dims=[2, 3]
    )

    logits_hv = model(
        images_hv
    )

    probs_hv = F.softmax(
        logits_hv,
        dim=1
    )

    probs_list.append(
        probs_hv
    )


    # --------------------------
    # TTA 平均
    # --------------------------

    final_probs = (
        torch.stack(
            probs_list,
            dim=0
        )
        .mean(
            dim=0
        )
    )

    return final_probs


# ============================================================
# 12. 5-Fold Ensemble 推理
# ============================================================

def predict_5fold(
    model_name
):

    print(
        "\n" + "=" * 60
    )

    print(
        f"开始预测: {model_name}"
    )

    print(
        "=" * 60
    )

    all_fold_probs = []


    for fold in range(
        1,
        6
    ):

        print(
            f"\n[{model_name}] "
            f"Fold {fold}/5"
        )


        # -----------------------
        # 创建模型
        # -----------------------

        model = create_model(
            model_name,
            num_classes=NUM_CLASSES
        )


        # -----------------------
        # 加载 checkpoint
        # -----------------------

        model_path = os.path.join(
            MODEL_DIR,
            f"{model_name}_fold{fold}.pth"
        )

        model = load_checkpoint(
            model,
            model_path
        )

        model.eval()


        # -----------------------
        # 当前 Fold 推理
        # -----------------------

        fold_probs = []


        with torch.no_grad():

            for batch_idx, (
                images,
                filenames
            ) in enumerate(
                test_loader
            ):

                images = images.to(
                    device,
                    non_blocking=True
                )


                probs = predict_tta(
                    model,
                    images
                )


                fold_probs.append(
                    probs.cpu()
                )


        fold_probs = torch.cat(
            fold_probs,
            dim=0
        )


        print(
            "Fold prediction shape:",
            tuple(
                fold_probs.shape
            )
        )


        all_fold_probs.append(
            fold_probs
        )


        # -----------------------
        # 清理显存
        # -----------------------

        del model
        del fold_probs

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    # ========================================================
    # 五折概率平均
    # ========================================================

    ensemble_probs = (
        torch.stack(
            all_fold_probs,
            dim=0
        )
        .mean(
            dim=0
        )
    )


    print(
        f"\n{model_name} "
        f"5-fold ensemble shape:",
        tuple(
            ensemble_probs.shape
        )
    )


    return ensemble_probs


# ============================================================
# 13. ResNet18 5-Fold + TTA
# ============================================================

resnet_probs = predict_5fold(
    "resnet18"
)


# ============================================================
# 14. EfficientNet-B0 5-Fold + TTA
# ============================================================

efficientnet_probs = predict_5fold(
    "efficientnet_b0"
)


# ============================================================
# 15. 检查输出
# ============================================================

print(
    "\nResNet shape:",
    tuple(
        resnet_probs.shape
    )
)

print(
    "EfficientNet shape:",
    tuple(
        efficientnet_probs.shape
    )
)


if (
    resnet_probs.shape
    != efficientnet_probs.shape
):

    raise ValueError(
        "两个模型的预测 Shape 不一致。"
    )


if (
    resnet_probs.shape[0]
    != len(submission)
):

    raise ValueError(
        "预测数量与 Submission 行数不一致。"
    )


# ============================================================
# 16. ResNet + EfficientNet Soft Voting
#
# 第一版使用 0.5 : 0.5
# ============================================================

RESNET_WEIGHT = 0.5
EFFICIENTNET_WEIGHT = 0.5


final_probs = (

    RESNET_WEIGHT
    * resnet_probs

    +

    EFFICIENTNET_WEIGHT
    * efficientnet_probs
)


# ============================================================
# 17. Argmax 获得最终类别
# ============================================================

final_preds = torch.argmax(
    final_probs,
    dim=1
).numpy()


predicted_classes = [

    idx_to_class[
        int(pred)
    ]

    for pred in final_preds
]


# ============================================================
# 18. 生成 Submission
# ============================================================

final_submission = (
    submission.copy()
)

final_submission[
    "Category"
] = predicted_classes


final_submission.to_csv(
    OUTPUT_PATH,
    index=False
)


# ============================================================
# 19. 最终检查
# ============================================================

print(
    "\n" + "=" * 60
)

print(
    "最终 Submission 生成完成"
)

print(
    "=" * 60
)


print(
    "\n保存位置:"
)

print(
    OUTPUT_PATH
)


print(
    "\nShape:"
)

print(
    final_submission.shape
)


print(
    "\nColumns:"
)

print(
    final_submission.columns.tolist()
)


print(
    "\nHead:"
)

print(
    final_submission.head(10)
)


print(
    "\n预测类别分布:"
)

print(
    final_submission[
        "Category"
    ].value_counts()
)


# ============================================================
# 20. 验证所有预测类别合法
# ============================================================

invalid_classes = set(
    final_submission[
        "Category"
    ].unique()
) - set(
    classes
)


if invalid_classes:

    raise ValueError(
        f"出现非法类别: {invalid_classes}"
    )


if len(
    final_submission
) != 378:

    print(
        "\n警告："
        "Submission 行数不是 378，"
        "请确认数据集是否发生变化。"
    )


print(
    "\n✅ 推理全部完成，可以提交 Kaggle。"
)

In [ ]:
import os

model_names = []

for fold in range(1, 6):
    model_names.append(f"resnet18_fold{fold}.pth")
    model_names.append(f"efficientnet_b0_fold{fold}.pth")

print("需要寻找的模型：", len(model_names))

found_models = {}

# 搜索输出目录与数据目录
search_roots = [
    str(ARTIFACT_DIR),
    str(DATA_DIR)
]

for search_root in search_roots:

    if not os.path.exists(search_root):
        continue

    print(f"\n正在搜索：{search_root}")

    for root, dirs, files in os.walk(search_root):

        for filename in files:

            if filename in model_names:

                full_path = os.path.join(
                    root,
                    filename
                )

                found_models[filename] = full_path

                print(
                    f"FOUND: {filename}\n"
                    f"       {full_path}"
                )

print("\n" + "=" * 60)

print(
    f"找到 {len(found_models)} / 10 个模型"
)

print("=" * 60)

missing = [
    name
    for name in model_names
    if name not in found_models
]

if missing:

    print("\n缺少：")

    for name in missing:
        print(" -", name)

else:

    print("\n✅ 10 个模型全部找到！")

In [ ]:
# ============================================================
# Plant Classification
# 5-Fold ResNet18 + EfficientNet-B0 + OOF Weight Search + TTA
# 一键训练 / 保存 / Ensemble / Submission
# ============================================================

import os
import gc
import random
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


# ============================================================
# 1. 路径
# ============================================================

DATA_ROOT = str(DATA_DIR)

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")

SUBMISSION_PATH = str(SAMPLE_SUBMISSION)

WORK_DIR = str(ARTIFACT_DIR)

os.makedirs(WORK_DIR, exist_ok=True)


# ============================================================
# 2. 基本参数
# ============================================================

SEED = 42
N_SPLITS = 5
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4

classes = [
    "Black-grass",
    "Common wheat",
    "Loose Silky-bent",
    "Scentless Mayweed",
    "Sugar beet"
]

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

idx_to_class = {
    idx: cls
    for cls, idx in class_to_idx.items()
}

NUM_CLASSES = len(classes)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("Device:", device)
print("=" * 70)


# ============================================================
# 3. 构建训练 DataFrame
# ============================================================

records = []

for class_name in classes:

    class_dir = os.path.join(
        TRAIN_DIR,
        class_name
    )

    for filename in os.listdir(class_dir):

        if filename.lower().endswith(
            (".png", ".jpg", ".jpeg", ".bmp", ".webp")
        ):

            records.append({
                "path": os.path.join(
                    class_dir,
                    filename
                ),
                "label": class_name
            })


df = pd.DataFrame(records)

print("\nTraining samples:", len(df))
print(df["label"].value_counts())


if len(df) != 500:
    print(
        "\n⚠️ 当前训练数据不是之前的500张，"
        "请确认数据集是否发生变化。"
    )


# ============================================================
# 4. Transform
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        20
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 5. Dataset
# ============================================================

class PlantDataset(Dataset):

    def __init__(
        self,
        dataframe,
        class_to_idx,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
        )

        self.class_to_idx = (
            class_to_idx
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.dataframe
        )


    def __getitem__(self, idx):

        row = (
            self.dataframe
            .iloc[idx]
        )

        image = Image.open(
            row["path"]
        ).convert("RGB")

        label = self.class_to_idx[
            row["label"]
        ]

        if self.transform:
            image = self.transform(
                image
            )

        return image, label


# ============================================================
# 6. 创建模型
# ============================================================

def create_model(
    model_name,
    pretrained=True
):

    if model_name == "resnet18":

        weights = (
            models.ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        model = models.resnet18(
            weights=weights
        )

        in_features = (
            model.fc.in_features
        )

        model.fc = nn.Linear(
            in_features,
            NUM_CLASSES
        )


    elif model_name == "efficientnet_b0":

        weights = (
            models.EfficientNet_B0_Weights.DEFAULT
            if pretrained
            else None
        )

        model = models.efficientnet_b0(
            weights=weights
        )

        in_features = (
            model.classifier[1]
            .in_features
        )

        model.classifier[1] = nn.Linear(
            in_features,
            NUM_CLASSES
        )


    else:

        raise ValueError(
            model_name
        )


    return model.to(device)


# ============================================================
# 7. Validation
# ============================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            total_loss += (
                loss.item()
                * images.size(0)
            )

            preds = outputs.argmax(
                dim=1
            )

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )


    val_loss = (
        total_loss /
        len(loader.dataset)
    )

    val_f1 = f1_score(
        labels_all,
        preds_all,
        average="macro"
    )

    val_acc = np.mean(
        np.array(preds_all)
        ==
        np.array(labels_all)
    )

    return (
        val_loss,
        val_acc,
        val_f1
    )


# ============================================================
# 8. TTA概率
# ============================================================

def tta_probs(
    model,
    images
):

    probs_list = []

    # 原图
    probs_list.append(
        F.softmax(
            model(images),
            dim=1
        )
    )

    # 水平翻转
    x = torch.flip(
        images,
        dims=[3]
    )

    probs_list.append(
        F.softmax(
            model(x),
            dim=1
        )
    )

    # 垂直翻转
    x = torch.flip(
        images,
        dims=[2]
    )

    probs_list.append(
        F.softmax(
            model(x),
            dim=1
        )
    )

    # 水平 + 垂直
    x = torch.flip(
        images,
        dims=[2, 3]
    )

    probs_list.append(
        F.softmax(
            model(x),
            dim=1
        )
    )

    return torch.stack(
        probs_list,
        dim=0
    ).mean(dim=0)


# ============================================================
# 9. 加载 checkpoint
# ============================================================

def load_checkpoint(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )

    model.load_state_dict(
        state
    )

    return model


# ============================================================
# 10. 单 Fold 训练
# ============================================================

def train_one_fold(
    model_name,
    fold,
    train_df,
    val_df
):

    print("\n" + "=" * 70)
    print(
        f"{model_name} | Fold {fold}"
    )
    print("=" * 70)

    set_seed(
        SEED + fold
    )

    train_dataset = PlantDataset(
        train_df,
        class_to_idx,
        train_transform
    )

    val_dataset = PlantDataset(
        val_df,
        class_to_idx,
        val_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = create_model(
        model_name,
        pretrained=True
    )

    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )
    )


    best_f1 = -1
    best_epoch = 0

    checkpoint_path = os.path.join(
        WORK_DIR,
        f"{model_name}_fold{fold}.pth"
    )


    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
                * images.size(0)
            )


        train_loss = (
            running_loss /
            len(train_dataset)
        )


        val_loss, val_acc, val_f1 = evaluate(
            model,
            val_loader,
            criterion
        )

        scheduler.step(
            val_f1
        )


        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Acc {val_acc:.4f} | "
            f"F1 {val_f1:.4f}"
        )


        if val_f1 > best_f1:

            best_f1 = val_f1
            best_epoch = epoch + 1

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"  -> Saved best "
                f"F1={best_f1:.4f}"
            )


    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return (
        best_f1,
        best_epoch,
        checkpoint_path
    )


# ============================================================
# 11. 建立统一5折
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

splits = list(
    skf.split(
        df,
        df["label"]
    )
)


# ============================================================
# 12. 训练两种模型
# ============================================================

all_results = {}


for model_name in [
    "resnet18",
    "efficientnet_b0"
]:

    model_results = []

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        splits,
        start=1
    ):

        train_fold_df = (
            df.iloc[train_idx]
            .reset_index(drop=True)
        )

        val_fold_df = (
            df.iloc[val_idx]
            .reset_index(drop=True)
        )

        best_f1, best_epoch, path = (
            train_one_fold(
                model_name,
                fold,
                train_fold_df,
                val_fold_df
            )
        )

        model_results.append({
            "fold": fold,
            "f1": best_f1,
            "epoch": best_epoch,
            "path": path
        })


    all_results[
        model_name
    ] = model_results


# ============================================================
# 13. 打印CV结果
# ============================================================

print("\n\n" + "=" * 70)
print("5-FOLD RESULTS")
print("=" * 70)


for model_name, results in all_results.items():

    scores = [
        x["f1"]
        for x in results
    ]

    print(
        f"\n{model_name}"
    )

    for x in results:

        print(
            f"Fold {x['fold']} | "
            f"F1={x['f1']:.4f} | "
            f"Epoch={x['epoch']}"
        )

    print(
        f"Mean F1 = "
        f"{np.mean(scores):.4f}"
    )

    print(
        f"Std F1  = "
        f"{np.std(scores):.4f}"
    )


# ============================================================
# 14. OOF概率
#
# 使用最佳checkpoint + TTA
# ============================================================

def generate_oof(
    model_name
):

    oof_probs = np.zeros(
        (
            len(df),
            NUM_CLASSES
        ),
        dtype=np.float32
    )

    oof_labels = np.zeros(
        len(df),
        dtype=np.int64
    )


    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        splits,
        start=1
    ):

        print(
            f"OOF {model_name} "
            f"Fold {fold}"
        )

        val_fold_df = (
            df.iloc[val_idx]
            .reset_index(drop=True)
        )

        val_dataset = PlantDataset(
            val_fold_df,
            class_to_idx,
            val_transform
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )


        model = create_model(
            model_name,
            pretrained=False
        )

        path = os.path.join(
            WORK_DIR,
            f"{model_name}_fold{fold}.pth"
        )

        model = load_checkpoint(
            model,
            path
        )

        model.eval()

        fold_probs = []
        fold_labels = []


        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(
                    device,
                    non_blocking=True
                )

                probs = tta_probs(
                    model,
                    images
                )

                fold_probs.append(
                    probs.cpu().numpy()
                )

                fold_labels.append(
                    labels.numpy()
                )


        fold_probs = np.concatenate(
            fold_probs,
            axis=0
        )

        fold_labels = np.concatenate(
            fold_labels,
            axis=0
        )


        oof_probs[
            val_idx
        ] = fold_probs

        oof_labels[
            val_idx
        ] = fold_labels


        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    return (
        oof_probs,
        oof_labels
    )


print("\nGenerating ResNet OOF...")
resnet_oof, oof_labels = generate_oof(
    "resnet18"
)

print("\nGenerating EfficientNet OOF...")
efficient_oof, _ = generate_oof(
    "efficientnet_b0"
)


# ============================================================
# 15. 单模型OOF F1
# ============================================================

resnet_oof_f1 = f1_score(
    oof_labels,
    resnet_oof.argmax(axis=1),
    average="macro"
)

efficient_oof_f1 = f1_score(
    oof_labels,
    efficient_oof.argmax(axis=1),
    average="macro"
)

print("\n" + "=" * 70)

print(
    f"ResNet OOF TTA F1: "
    f"{resnet_oof_f1:.4f}"
)

print(
    f"EfficientNet OOF TTA F1: "
    f"{efficient_oof_f1:.4f}"
)


# ============================================================
# 16. 自动搜索融合权重
#
# alpha = EfficientNet权重
# 1-alpha = ResNet权重
# ============================================================

best_alpha = None
best_oof_f1 = -1


print("\nFusion weight search:")

for alpha in np.arange(
    0.0,
    1.01,
    0.05
):

    fused = (
        (1 - alpha)
        * resnet_oof

        +

        alpha
        * efficient_oof
    )

    preds = fused.argmax(
        axis=1
    )

    score = f1_score(
        oof_labels,
        preds,
        average="macro"
    )

    print(
        f"EfficientNet weight "
        f"{alpha:.2f} | "
        f"OOF F1 {score:.4f}"
    )

    if score > best_oof_f1:

        best_oof_f1 = score
        best_alpha = float(alpha)


print("\n" + "=" * 70)

print(
    f"BEST EfficientNet Weight: "
    f"{best_alpha:.2f}"
)

print(
    f"BEST ResNet Weight: "
    f"{1-best_alpha:.2f}"
)

print(
    f"BEST Fusion OOF F1: "
    f"{best_oof_f1:.4f}"
)

print("=" * 70)


# ============================================================
# 17. Test Dataset
# ============================================================

submission = pd.read_csv(
    SUBMISSION_PATH
)

test_filenames = (
    submission["ID"]
    .astype(str)
    .tolist()
)


# 自动递归寻找测试图片
test_file_map = {}

for root, dirs, files in os.walk(
    TEST_DIR
):

    for filename in files:

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            test_file_map[
                filename
            ] = os.path.join(
                root,
                filename
            )


missing = [
    x
    for x in test_filenames
    if x not in test_file_map
]

if missing:

    raise FileNotFoundError(
        f"找不到 {len(missing)} 张测试图片。"
    )


class PlantTestDataset(Dataset):

    def __init__(
        self,
        filenames,
        file_map,
        transform
    ):

        self.filenames = filenames
        self.file_map = file_map
        self.transform = transform


    def __len__(self):

        return len(
            self.filenames
        )


    def __getitem__(self, idx):

        filename = (
            self.filenames[idx]
        )

        image = Image.open(
            self.file_map[
                filename
            ]
        ).convert(
            "RGB"
        )

        image = self.transform(
            image
        )

        return (
            image,
            filename
        )


test_dataset = PlantTestDataset(
    test_filenames,
    test_file_map,
    val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "\nTest samples:",
    len(test_dataset)
)


# ============================================================
# 18. Test 5-Fold + TTA
# ============================================================

def predict_test_5fold(
    model_name
):

    fold_predictions = []


    for fold in range(
        1,
        N_SPLITS + 1
    ):

        print(
            f"Test inference | "
            f"{model_name} | "
            f"Fold {fold}"
        )

        model = create_model(
            model_name,
            pretrained=False
        )

        path = os.path.join(
            WORK_DIR,
            f"{model_name}_fold{fold}.pth"
        )

        model = load_checkpoint(
            model,
            path
        )

        model.eval()

        probabilities = []


        with torch.no_grad():

            for images, _ in test_loader:

                images = images.to(
                    device,
                    non_blocking=True
                )

                probs = tta_probs(
                    model,
                    images
                )

                probabilities.append(
                    probs.cpu().numpy()
                )


        probabilities = np.concatenate(
            probabilities,
            axis=0
        )

        fold_predictions.append(
            probabilities
        )


        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    return np.mean(
        np.stack(
            fold_predictions,
            axis=0
        ),
        axis=0
    )


print("\nPredicting ResNet...")
resnet_test_probs = (
    predict_test_5fold(
        "resnet18"
    )
)

print("\nPredicting EfficientNet...")
efficient_test_probs = (
    predict_test_5fold(
        "efficientnet_b0"
    )
)


# ============================================================
# 19. 使用OOF自动得到的最佳权重融合
# ============================================================

final_probs = (

    (1 - best_alpha)
    * resnet_test_probs

    +

    best_alpha
    * efficient_test_probs
)


final_predictions = (
    final_probs.argmax(
        axis=1
    )
)


predicted_classes = [

    idx_to_class[
        int(x)
    ]

    for x in final_predictions
]


# ============================================================
# 20. Submission
# ============================================================

final_submission = (
    submission.copy()
)

final_submission[
    "Category"
] = predicted_classes


OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "submission_ensemble_tta.csv"
)


final_submission.to_csv(
    OUTPUT_PATH,
    index=False
)


print("\n" + "=" * 70)

print(
    "SUBMISSION CREATED"
)

print("=" * 70)

print(
    "Path:",
    OUTPUT_PATH
)

print(
    "Shape:",
    final_submission.shape
)

print("\nPrediction Distribution:")

print(
    final_submission[
        "Category"
    ].value_counts()
)

print("\nHead:")

print(
    final_submission.head()
)


# ============================================================
# 21. 打包10个Checkpoint
# ============================================================

ZIP_PATH = os.path.join(
    WORK_DIR,
    "plant_fold_models.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for model_name in [
        "resnet18",
        "efficientnet_b0"
    ]:

        for fold in range(
            1,
            N_SPLITS + 1
        ):

            filename = (
                f"{model_name}_fold{fold}.pth"
            )

            path = os.path.join(
                WORK_DIR,
                filename
            )

            if os.path.exists(path):

                z.write(
                    path,
                    arcname=filename
                )


print(
    "\nCheckpoint ZIP:",
    ZIP_PATH
)


# ============================================================
# 22. 最终模型检查
# ============================================================

print("\nSaved checkpoints:")

for model_name in [
    "resnet18",
    "efficientnet_b0"
]:

    for fold in range(
        1,
        6
    ):

        path = os.path.join(
            WORK_DIR,
            f"{model_name}_fold{fold}.pth"
        )

        print(
            os.path.basename(path),
            "✅"
            if os.path.exists(path)
            else "❌"
        )


print("\n" + "=" * 70)

print("✅ ALL DONE")

print(
    "\n提交文件：" + str(ARTIFACT_DIR / "submission_ensemble_tta.csv")
)

print(
    "\n模型备份：" + str(ARTIFACT_DIR / "plant_fold_models.zip")
)

print("=" * 70)

In [ ]:
# ============================================================
# Experiment 6
# Black-grass vs Loose Silky-bent
# 5-Fold EfficientNet-B0 Binary Expert
#
# 功能：
# 1. 自动读取两个困难类别，共200张图片
# 2. 320×320输入
# 3. EfficientNet-B0 ImageNet预训练
# 4. 5-Fold Stratified CV
# 5. 保存5个最佳二分类模型
# 6. 生成OOF概率
# 7. 计算Plain OOF F1和TTA OOF F1
# 8. 保存OOF CSV和模型ZIP
# ============================================================

import os
import gc
import random
import zipfile
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. 路径
# ============================================================

DATA_ROOT = str(DATA_DIR)

TRAIN_DIR = os.path.join(
    DATA_ROOT,
    "train"
)

WORK_DIR = str(ARTIFACT_DIR)

os.makedirs(
    WORK_DIR,
    exist_ok=True
)


# ============================================================
# 2. 参数
# ============================================================

SEED = 42

N_SPLITS = 5

EPOCHS = 15

BATCH_SIZE = 16

LR = 1e-4

WEIGHT_DECAY = 1e-4

IMAGE_SIZE = 320


# 这里只研究两个困难类别
BINARY_CLASSES = [
    "Black-grass",
    "Loose Silky-bent"
]

binary_class_to_idx = {
    "Black-grass": 0,
    "Loose Silky-bent": 1
}

binary_idx_to_class = {
    0: "Black-grass",
    1: "Loose Silky-bent"
}


# ============================================================
# 3. Seed
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(SEED)


# ============================================================
# 4. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)

print(
    "Device:",
    device
)

print("=" * 70)


# ============================================================
# 5. 建立二分类 DataFrame
# ============================================================

records = []

for class_name in BINARY_CLASSES:

    class_dir = os.path.join(
        TRAIN_DIR,
        class_name
    )

    if not os.path.exists(class_dir):

        raise FileNotFoundError(
            class_dir
        )

    for filename in os.listdir(
        class_dir
    ):

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            records.append({

                "path": os.path.join(
                    class_dir,
                    filename
                ),

                "filename": filename,

                "label_name": class_name,

                "label": binary_class_to_idx[
                    class_name
                ]

            })


binary_df = pd.DataFrame(
    records
)


print(
    "\nBinary samples:",
    len(binary_df)
)

print(
    "\nClass distribution:"
)

print(
    binary_df[
        "label_name"
    ].value_counts()
)


if len(binary_df) != 200:

    print(
        "\n⚠️ 预期200张图片，"
        f"当前找到 {len(binary_df)} 张。"
    )


# ============================================================
# 6. Transform
#
# 不使用 RandomResizedCrop
# 尽量保留完整叶片形态
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        20
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 7. Dataset
# ============================================================

class BinaryPlantDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        image = Image.open(
            row["path"]
        ).convert(
            "RGB"
        )

        label = int(
            row["label"]
        )

        if self.transform:

            image = self.transform(
                image
            )

        return (
            image,
            label
        )


# ============================================================
# 8. 创建 Binary EfficientNet-B0
# ============================================================

def create_binary_model(
    pretrained=True
):

    weights = (
        models
        .EfficientNet_B0_Weights
        .DEFAULT
        if pretrained
        else None
    )

    model = (
        models
        .efficientnet_b0(
            weights=weights
        )
    )

    in_features = (
        model
        .classifier[1]
        .in_features
    )

    model.classifier[1] = (
        nn.Linear(
            in_features,
            2
        )
    )

    return model.to(
        device
    )


# ============================================================
# 9. 普通 Validation
# ============================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0

    all_labels = []

    all_preds = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )

            total_loss += (
                loss.item()
                * images.size(0)
            )

            preds = logits.argmax(
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    val_loss = (
        total_loss
        /
        len(loader.dataset)
    )

    val_acc = accuracy_score(
        all_labels,
        all_preds
    )

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return (
        val_loss,
        val_acc,
        val_f1
    )


# ============================================================
# 10. TTA概率
# ============================================================

def binary_tta_probs(
    model,
    images
):

    probs_list = []


    # Original
    logits = model(
        images
    )

    probs_list.append(
        F.softmax(
            logits,
            dim=1
        )
    )


    # Horizontal
    x = torch.flip(
        images,
        dims=[3]
    )

    probs_list.append(
        F.softmax(
            model(x),
            dim=1
        )
    )


    # Vertical
    x = torch.flip(
        images,
        dims=[2]
    )

    probs_list.append(
        F.softmax(
            model(x),
            dim=1
        )
    )


    # Horizontal + Vertical
    x = torch.flip(
        images,
        dims=[
            2,
            3
        ]
    )

    probs_list.append(
        F.softmax(
            model(x),
            dim=1
        )
    )


    return (
        torch
        .stack(
            probs_list,
            dim=0
        )
        .mean(
            dim=0
        )
    )


# ============================================================
# 11. checkpoint loader
# ============================================================

def load_checkpoint(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )

    model.load_state_dict(
        state
    )

    return model


# ============================================================
# 12. 5-Fold
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

splits = list(
    skf.split(
        binary_df,
        binary_df[
            "label"
        ]
    )
)


# ============================================================
# 13. 单Fold训练
# ============================================================

def train_one_fold(
    fold,
    train_idx,
    val_idx
):

    print(
        "\n" + "=" * 70
    )

    print(
        f"Binary Expert | Fold {fold}"
    )

    print(
        "=" * 70
    )


    set_seed(
        SEED + fold
    )


    train_df = (
        binary_df
        .iloc[
            train_idx
        ]
        .reset_index(
            drop=True
        )
    )

    val_df = (
        binary_df
        .iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    train_dataset = (
        BinaryPlantDataset(
            train_df,
            train_transform
        )
    )

    val_dataset = (
        BinaryPlantDataset(
            val_df,
            val_transform
        )
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=2,

        pin_memory=True
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=2,

        pin_memory=True
    )


    model = create_binary_model(
        pretrained=True
    )


    criterion = (
        nn.CrossEntropyLoss()
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=LR,

            weight_decay=WEIGHT_DECAY
        )
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(

            optimizer,

            mode="max",

            factor=0.5,

            patience=2
        )
    )


    best_f1 = -1.0

    best_epoch = 0


    checkpoint_path = (
        os.path.join(

            WORK_DIR,

            f"binary_expert_fold{fold}.pth"
        )
    )


    for epoch in range(
        EPOCHS
    ):

        model.train()

        running_loss = 0.0


        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad()


            logits = model(
                images
            )


            loss = criterion(
                logits,
                labels
            )


            loss.backward()


            optimizer.step()


            running_loss += (
                loss.item()
                * images.size(0)
            )


        train_loss = (
            running_loss
            /
            len(
                train_dataset
            )
        )


        (
            val_loss,
            val_acc,
            val_f1
        ) = evaluate(

            model,

            val_loader,

            criterion
        )


        scheduler.step(
            val_f1
        )


        print(

            f"Epoch "
            f"[{epoch+1:02d}/{EPOCHS}] | "

            f"Train Loss "
            f"{train_loss:.4f} | "

            f"Val Loss "
            f"{val_loss:.4f} | "

            f"Acc "
            f"{val_acc:.4f} | "

            f"F1 "
            f"{val_f1:.4f}"
        )


        if val_f1 > best_f1:

            best_f1 = val_f1

            best_epoch = (
                epoch + 1
            )

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"  -> Saved best "
                f"F1={best_f1:.4f}"
            )


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {

        "fold": fold,

        "best_f1": best_f1,

        "best_epoch": best_epoch,

        "path": checkpoint_path
    }


# ============================================================
# 14. 正式训练5折
# ============================================================

fold_results = []


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    result = train_one_fold(

        fold,
        train_idx,
        val_idx
    )

    fold_results.append(
        result
    )


# ============================================================
# 15. 打印 Fold 结果
# ============================================================

fold_scores = [

    x[
        "best_f1"
    ]

    for x in fold_results
]


print(
    "\n\n" + "=" * 70
)

print(
    "BINARY EXPERT 5-FOLD RESULTS"
)

print(
    "=" * 70
)


for result in fold_results:

    print(

        f"Fold "
        f"{result['fold']} | "

        f"F1="
        f"{result['best_f1']:.4f} | "

        f"Best Epoch="
        f"{result['best_epoch']}"
    )


print(
    "\nMean F1:",
    np.mean(
        fold_scores
    )
)

print(
    "Std F1:",
    np.std(
        fold_scores
    )
)


# ============================================================
# 16. 生成Plain OOF + TTA OOF
# ============================================================

oof_plain_probs = np.zeros(
    (
        len(binary_df),
        2
    ),
    dtype=np.float32
)

oof_tta_probs = np.zeros(
    (
        len(binary_df),
        2
    ),
    dtype=np.float32
)


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    print(
        f"\nGenerating OOF | "
        f"Fold {fold}"
    )


    val_df = (
        binary_df
        .iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    val_dataset = (
        BinaryPlantDataset(
            val_df,
            val_transform
        )
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=2,

        pin_memory=True
    )


    model = create_binary_model(
        pretrained=False
    )


    checkpoint_path = (
        os.path.join(

            WORK_DIR,

            f"binary_expert_fold{fold}.pth"
        )
    )


    model = load_checkpoint(
        model,
        checkpoint_path
    )

    model.eval()


    plain_probs_list = []

    tta_probs_list = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )


            # Plain
            logits = model(
                images
            )

            plain_probs = F.softmax(
                logits,
                dim=1
            )


            # TTA
            tta_probs = (
                binary_tta_probs(
                    model,
                    images
                )
            )


            plain_probs_list.append(
                plain_probs
                .cpu()
                .numpy()
            )

            tta_probs_list.append(
                tta_probs
                .cpu()
                .numpy()
            )


    fold_plain = np.concatenate(
        plain_probs_list,
        axis=0
    )

    fold_tta = np.concatenate(
        tta_probs_list,
        axis=0
    )


    oof_plain_probs[
        val_idx
    ] = fold_plain


    oof_tta_probs[
        val_idx
    ] = fold_tta


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 17. OOF评估
# ============================================================

y_true = (
    binary_df[
        "label"
    ].values
)


plain_preds = (
    oof_plain_probs
    .argmax(
        axis=1
    )
)


tta_preds = (
    oof_tta_probs
    .argmax(
        axis=1
    )
)


plain_f1 = f1_score(

    y_true,

    plain_preds,

    average="macro"
)


tta_f1 = f1_score(

    y_true,

    tta_preds,

    average="macro"
)


plain_acc = accuracy_score(

    y_true,

    plain_preds
)


tta_acc = accuracy_score(

    y_true,

    tta_preds
)


print(
    "\n" + "=" * 70
)

print(
    "BINARY OOF RESULTS"
)

print(
    "=" * 70
)


print(
    f"Plain OOF Accuracy: "
    f"{plain_acc:.4f}"
)

print(
    f"Plain OOF Macro F1: "
    f"{plain_f1:.4f}"
)

print()

print(
    f"TTA OOF Accuracy: "
    f"{tta_acc:.4f}"
)

print(
    f"TTA OOF Macro F1: "
    f"{tta_f1:.4f}"
)


# ============================================================
# 18. Classification Report
# ============================================================

print(
    "\nTTA Classification Report:\n"
)

print(

    classification_report(

        y_true,

        tta_preds,

        target_names=BINARY_CLASSES,

        digits=4
    )
)


# ============================================================
# 19. Confusion Matrix
# ============================================================

cm = confusion_matrix(

    y_true,

    tta_preds
)


print(
    "\nTTA Confusion Matrix:"
)

print(
    cm
)


# ============================================================
# 20. 保存OOF CSV
# ============================================================

oof_df = (
    binary_df[
        [
            "filename",
            "path",
            "label_name",
            "label"
        ]
    ]
    .copy()
)


oof_df[
    "p_black_grass"
] = (
    oof_tta_probs[
        :,
        0
    ]
)


oof_df[
    "p_loose_silky_bent"
] = (
    oof_tta_probs[
        :,
        1
    ]
)


oof_df[
    "pred"
] = tta_preds


oof_df[
    "pred_name"
] = [

    binary_idx_to_class[
        int(x)
    ]

    for x in tta_preds
]


OOF_PATH = os.path.join(

    WORK_DIR,

    "binary_expert_oof.csv"
)


oof_df.to_csv(

    OOF_PATH,

    index=False
)


print(
    "\nOOF saved:"
)

print(
    OOF_PATH
)


# ============================================================
# 21. 保存实验摘要
# ============================================================

summary_df = pd.DataFrame({

    "metric": [

        "mean_fold_f1",
        "std_fold_f1",
        "plain_oof_f1",
        "tta_oof_f1",
        "plain_oof_acc",
        "tta_oof_acc"
    ],

    "value": [

        np.mean(
            fold_scores
        ),

        np.std(
            fold_scores
        ),

        plain_f1,

        tta_f1,

        plain_acc,

        tta_acc
    ]
})


SUMMARY_PATH = os.path.join(

    WORK_DIR,

    "binary_expert_summary.csv"
)


summary_df.to_csv(

    SUMMARY_PATH,

    index=False
)


# ============================================================
# 22. 打包5个Binary模型
# ============================================================

ZIP_PATH = os.path.join(

    WORK_DIR,

    "binary_expert_models.zip"
)


with zipfile.ZipFile(

    ZIP_PATH,

    "w",

    zipfile.ZIP_DEFLATED

) as z:

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        filename = (
            f"binary_expert_fold{fold}.pth"
        )

        path = os.path.join(
            WORK_DIR,
            filename
        )

        if os.path.exists(
            path
        ):

            z.write(

                path,

                arcname=filename
            )


# ============================================================
# 23. 最终检查
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "BINARY EXPERT TRAINING DONE"
)

print(
    "=" * 70
)


print(
    "\nSaved models:"
)


for fold in range(
    1,
    6
):

    path = os.path.join(

        WORK_DIR,

        f"binary_expert_fold{fold}.pth"
    )

    print(

        os.path.basename(
            path
        ),

        "✅"
        if os.path.exists(
            path
        )
        else "❌"
    )


print(
    "\nOOF CSV:"
)

print(
    OOF_PATH
)


print(
    "\nSummary:"
)

print(
    SUMMARY_PATH
)


print(
    "\nModel ZIP:"
)

print(
    ZIP_PATH
)


print(
    "\n✅ Experiment 6 finished."
)

In [ ]:
# ============================================================
# Experiment 7
# Black-grass vs Loose Silky-bent
# Foreground Crop + EfficientNet-B0 + 5-Fold
#
# 核心变量：
# 原始图 -> Excess Green植物前景提取 -> Square Crop -> 320x320
#
# 不使用 TTA
# ============================================================

import os
import gc
import random
import zipfile
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. 路径
# ============================================================

DATA_ROOT = str(DATA_DIR)

TRAIN_DIR = os.path.join(
    DATA_ROOT,
    "train"
)

WORK_DIR = str(ARTIFACT_DIR)

os.makedirs(
    WORK_DIR,
    exist_ok=True
)


# ============================================================
# 2. 参数
# ============================================================

SEED = 42

N_SPLITS = 5

EPOCHS = 15

BATCH_SIZE = 16

IMAGE_SIZE = 320

LR = 1e-4

WEIGHT_DECAY = 1e-4


# 上一轮真实Plain OOF结果
BASELINE_BINARY_F1 = 0.7647


BINARY_CLASSES = [
    "Black-grass",
    "Loose Silky-bent"
]

class_to_idx = {
    "Black-grass": 0,
    "Loose Silky-bent": 1
}

idx_to_class = {
    0: "Black-grass",
    1: "Loose Silky-bent"
}


# ============================================================
# 3. Seed
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(SEED)


# ============================================================
# 4. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("Device:", device)
print("=" * 70)


# ============================================================
# 5. 建立200张困难样本DataFrame
# ============================================================

records = []

for class_name in BINARY_CLASSES:

    class_dir = os.path.join(
        TRAIN_DIR,
        class_name
    )

    if not os.path.exists(class_dir):

        raise FileNotFoundError(
            class_dir
        )

    # 故意保持与上一轮相同的os.listdir方式
    for filename in os.listdir(
        class_dir
    ):

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            records.append({
                "path": os.path.join(
                    class_dir,
                    filename
                ),

                "filename": filename,

                "label_name": class_name,

                "label": class_to_idx[
                    class_name
                ]
            })


df = pd.DataFrame(
    records
)


print(
    "\nSamples:",
    len(df)
)

print(
    "\nClass distribution:"
)

print(
    df["label_name"]
    .value_counts()
)


# ============================================================
# 6. Excess Green前景裁剪
# ============================================================

def foreground_crop(
    image,
    margin=0.15,
    exg_threshold=20
):
    """
    使用 Excess Green:
        ExG = 2G - R - B

    提取绿色植物主体，得到bbox后扩张margin，
    再转成square crop。

    如果前景检测失败，自动返回原图。
    """

    image = image.convert("RGB")

    arr = np.asarray(
        image
    ).astype(
        np.float32
    )

    r = arr[:, :, 0]
    g = arr[:, :, 1]
    b = arr[:, :, 2]

    exg = (
        2 * g
        - r
        - b
    )

    # 双重条件降低泥土等背景误检
    mask = (
        (exg > exg_threshold)
        &
        (g > r + 5)
        &
        (g > b + 5)
    )


    height, width = mask.shape

    ys, xs = np.where(
        mask
    )


    # -------------------------------------
    # 检测失败时保留原图
    # -------------------------------------

    min_pixels = max(
        20,
        int(
            height
            * width
            * 0.002
        )
    )

    if len(xs) < min_pixels:

        return image


    # -------------------------------------
    # 使用1%~99%分位数
    # 排除少量孤立绿色噪点
    # -------------------------------------

    x1 = int(
        np.percentile(
            xs,
            1
        )
    )

    x2 = int(
        np.percentile(
            xs,
            99
        )
    )

    y1 = int(
        np.percentile(
            ys,
            1
        )
    )

    y2 = int(
        np.percentile(
            ys,
            99
        )
    )


    box_width = (
        x2 - x1 + 1
    )

    box_height = (
        y2 - y1 + 1
    )


    if (
        box_width <= 2
        or
        box_height <= 2
    ):

        return image


    # -------------------------------------
    # 建立Square Crop
    # -------------------------------------

    center_x = (
        x1 + x2
    ) / 2.0

    center_y = (
        y1 + y2
    ) / 2.0


    side = max(
        box_width,
        box_height
    )


    # 四周增加margin
    side = side * (
        1
        + 2 * margin
    )


    # 防止crop过小
    side = max(
        side,
        min(
            width,
            height
        ) * 0.20
    )


    # 不超过图像可用最大范围
    side = min(
        side,
        max(
            width,
            height
        )
    )


    side = int(
        round(side)
    )


    # -------------------------------------
    # Square bbox
    # -------------------------------------

    left = int(
        round(
            center_x
            - side / 2
        )
    )

    top = int(
        round(
            center_y
            - side / 2
        )
    )


    # 将bbox移动回图片内
    left = max(
        0,
        min(
            left,
            width - side
        )
    )

    top = max(
        0,
        min(
            top,
            height - side
        )
    )


    right = min(
        width,
        left + side
    )

    bottom = min(
        height,
        top + side
    )


    # 再次安全检查
    if (
        right <= left
        or
        bottom <= top
    ):

        return image


    crop = image.crop(
        (
            left,
            top,
            right,
            bottom
        )
    )

    return crop


# ============================================================
# 7. 检查前景裁剪实际效果
# ============================================================

area_ratios = []

for path in df["path"]:

    img = Image.open(
        path
    ).convert(
        "RGB"
    )

    crop = foreground_crop(
        img
    )

    original_area = (
        img.width
        * img.height
    )

    crop_area = (
        crop.width
        * crop.height
    )

    area_ratios.append(
        crop_area
        /
        original_area
    )


area_ratios = np.array(
    area_ratios
)


print(
    "\nForeground Crop Statistics"
)

print(
    "Mean crop/original area:",
    area_ratios.mean()
)

print(
    "Median crop/original area:",
    np.median(
        area_ratios
    )
)

print(
    "Min crop/original area:",
    area_ratios.min()
)

print(
    "Max crop/original area:",
    area_ratios.max()
)


# ============================================================
# 8. 可视化4组 Original vs Crop
# ============================================================

preview_indices = [
    0,
    25,
    100,
    125
]

fig, axes = plt.subplots(
    4,
    2,
    figsize=(8, 14)
)


for row_idx, sample_idx in enumerate(
    preview_indices
):

    sample = df.iloc[
        sample_idx
    ]

    img = Image.open(
        sample["path"]
    ).convert(
        "RGB"
    )

    crop = foreground_crop(
        img
    )


    axes[
        row_idx,
        0
    ].imshow(
        img
    )

    axes[
        row_idx,
        0
    ].set_title(
        f"Original - {sample['label_name']}"
    )

    axes[
        row_idx,
        0
    ].axis(
        "off"
    )


    axes[
        row_idx,
        1
    ].imshow(
        crop
    )

    axes[
        row_idx,
        1
    ].set_title(
        "Foreground Crop"
    )

    axes[
        row_idx,
        1
    ].axis(
        "off"
    )


plt.tight_layout()

PREVIEW_PATH = os.path.join(
    WORK_DIR,
    "foreground_crop_preview.png"
)

plt.savefig(
    PREVIEW_PATH,
    dpi=150
)

plt.show()

print(
    "\nPreview saved:",
    PREVIEW_PATH
)


# ============================================================
# 9. Transform
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        20
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 10. Dataset
# ============================================================

class ForegroundPlantDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        image = Image.open(
            row["path"]
        ).convert(
            "RGB"
        )

        # 核心变化就在这里
        image = foreground_crop(
            image
        )

        label = int(
            row["label"]
        )


        if self.transform:

            image = self.transform(
                image
            )


        return (
            image,
            label
        )


# ============================================================
# 11. 模型
# ============================================================

def create_model(
    pretrained=True
):

    weights = (
        models
        .EfficientNet_B0_Weights
        .DEFAULT
        if pretrained
        else None
    )


    model = models.efficientnet_b0(
        weights=weights
    )


    in_features = (
        model
        .classifier[1]
        .in_features
    )


    model.classifier[1] = (
        nn.Linear(
            in_features,
            2
        )
    )


    return model.to(
        device
    )


# ============================================================
# 12. Evaluate
# ============================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_labels = []

    all_preds = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            logits = model(
                images
            )


            loss = criterion(
                logits,
                labels
            )


            running_loss += (
                loss.item()
                * images.size(0)
            )


            preds = logits.argmax(
                dim=1
            )


            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    val_loss = (
        running_loss
        /
        len(
            loader.dataset
        )
    )


    val_acc = accuracy_score(
        all_labels,
        all_preds
    )


    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )


    return (
        val_loss,
        val_acc,
        val_f1
    )


# ============================================================
# 13. Checkpoint Loader
# ============================================================

def load_checkpoint(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )


    model.load_state_dict(
        state
    )


    return model


# ============================================================
# 14. 5-Fold
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


splits = list(
    skf.split(
        df,
        df["label"]
    )
)


# ============================================================
# 15. 单Fold训练
# ============================================================

def train_one_fold(
    fold,
    train_idx,
    val_idx
):

    print(
        "\n" + "=" * 70
    )

    print(
        f"Foreground Binary Expert | Fold {fold}"
    )

    print(
        "=" * 70
    )


    set_seed(
        SEED + fold
    )


    train_fold_df = (
        df.iloc[
            train_idx
        ]
        .reset_index(
            drop=True
        )
    )


    val_fold_df = (
        df.iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    train_dataset = (
        ForegroundPlantDataset(
            train_fold_df,
            train_transform
        )
    )


    val_dataset = (
        ForegroundPlantDataset(
            val_fold_df,
            val_transform
        )
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    model = create_model(
        pretrained=True
    )


    criterion = (
        nn.CrossEntropyLoss()
    )


    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )
    )


    best_f1 = -1.0

    best_epoch = 0


    checkpoint_path = os.path.join(
        WORK_DIR,
        f"foreground_binary_fold{fold}.pth"
    )


    for epoch in range(
        EPOCHS
    ):

        model.train()

        running_loss = 0.0


        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad()


            logits = model(
                images
            )


            loss = criterion(
                logits,
                labels
            )


            loss.backward()

            optimizer.step()


            running_loss += (
                loss.item()
                * images.size(0)
            )


        train_loss = (
            running_loss
            /
            len(
                train_dataset
            )
        )


        (
            val_loss,
            val_acc,
            val_f1
        ) = evaluate(
            model,
            val_loader,
            criterion
        )


        scheduler.step(
            val_f1
        )


        print(
            f"Epoch "
            f"[{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Acc {val_acc:.4f} | "
            f"F1 {val_f1:.4f}"
        )


        if val_f1 > best_f1:

            best_f1 = val_f1

            best_epoch = (
                epoch + 1
            )


            torch.save(
                model.state_dict(),
                checkpoint_path
            )


            print(
                f"  -> Saved best "
                f"F1={best_f1:.4f}"
            )


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {
        "fold": fold,
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "path": checkpoint_path
    }


# ============================================================
# 16. 训练5折
# ============================================================

results = []


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    result = train_one_fold(
        fold,
        train_idx,
        val_idx
    )

    results.append(
        result
    )


# ============================================================
# 17. Fold结果
# ============================================================

fold_scores = [
    x["best_f1"]
    for x in results
]


print(
    "\n\n" + "=" * 70
)

print(
    "FOREGROUND BINARY 5-FOLD RESULTS"
)

print(
    "=" * 70
)


for result in results:

    print(
        f"Fold {result['fold']} | "
        f"F1={result['best_f1']:.4f} | "
        f"Best Epoch={result['best_epoch']}"
    )


print(
    "\nMean Best-Fold F1:",
    np.mean(
        fold_scores
    )
)

print(
    "Std:",
    np.std(
        fold_scores
    )
)


# ============================================================
# 18. 重新加载最佳checkpoint生成Plain OOF
# ============================================================

oof_probs = np.zeros(
    (
        len(df),
        2
    ),
    dtype=np.float32
)


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    print(
        f"\nGenerating OOF "
        f"Fold {fold}"
    )


    val_fold_df = (
        df.iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    val_dataset = (
        ForegroundPlantDataset(
            val_fold_df,
            val_transform
        )
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    model = create_model(
        pretrained=False
    )


    model_path = os.path.join(
        WORK_DIR,
        f"foreground_binary_fold{fold}.pth"
    )


    model = load_checkpoint(
        model,
        model_path
    )


    model.eval()


    fold_probs = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )


            logits = model(
                images
            )


            probs = torch.softmax(
                logits,
                dim=1
            )


            fold_probs.append(
                probs
                .cpu()
                .numpy()
            )


    fold_probs = np.concatenate(
        fold_probs,
        axis=0
    )


    oof_probs[
        val_idx
    ] = fold_probs


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 19. 最终OOF指标
# ============================================================

y_true = (
    df["label"]
    .values
)


oof_preds = (
    oof_probs
    .argmax(
        axis=1
    )
)


oof_accuracy = accuracy_score(
    y_true,
    oof_preds
)


oof_f1 = f1_score(
    y_true,
    oof_preds,
    average="macro"
)


print(
    "\n" + "=" * 70
)

print(
    "FOREGROUND CROP OOF RESULTS"
)

print(
    "=" * 70
)


print(
    f"Accuracy: "
    f"{oof_accuracy:.4f}"
)

print(
    f"Macro F1: "
    f"{oof_f1:.4f}"
)


print(
    "\nPrevious Binary Baseline:"
)

print(
    f"{BASELINE_BINARY_F1:.4f}"
)


delta = (
    oof_f1
    -
    BASELINE_BINARY_F1
)


print(
    "\nDelta:"
)

print(
    f"{delta:+.4f}"
)


# ============================================================
# 20. Classification Report
# ============================================================

print(
    "\nClassification Report:\n"
)


print(
    classification_report(
        y_true,
        oof_preds,
        target_names=BINARY_CLASSES,
        digits=4
    )
)


# ============================================================
# 21. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    oof_preds
)


print(
    "\nConfusion Matrix:"
)

print(
    cm
)


# ============================================================
# 22. 保存 OOF
# ============================================================

oof_df = (
    df[
        [
            "filename",
            "path",
            "label_name",
            "label"
        ]
    ]
    .copy()
)


oof_df[
    "p_black_grass"
] = (
    oof_probs[
        :,
        0
    ]
)


oof_df[
    "p_loose_silky_bent"
] = (
    oof_probs[
        :,
        1
    ]
)


oof_df[
    "prediction"
] = (
    oof_preds
)


oof_df[
    "prediction_name"
] = [
    idx_to_class[
        int(x)
    ]
    for x in oof_preds
]


OOF_PATH = os.path.join(
    WORK_DIR,
    "foreground_binary_oof.csv"
)


oof_df.to_csv(
    OOF_PATH,
    index=False
)


# ============================================================
# 23. 保存实验摘要
# ============================================================

summary_df = pd.DataFrame({

    "metric": [
        "baseline_binary_f1",
        "foreground_oof_f1",
        "delta",
        "foreground_accuracy",
        "mean_best_fold_f1",
        "std_best_fold_f1"
    ],

    "value": [
        BASELINE_BINARY_F1,
        oof_f1,
        delta,
        oof_accuracy,
        np.mean(
            fold_scores
        ),
        np.std(
            fold_scores
        )
    ]
})


SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "foreground_binary_summary.csv"
)


summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 24. 打包模型
# ============================================================

ZIP_PATH = os.path.join(
    WORK_DIR,
    "foreground_binary_models.zip"
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for fold in range(
        1,
        6
    ):

        filename = (
            f"foreground_binary_fold{fold}.pth"
        )

        path = os.path.join(
            WORK_DIR,
            filename
        )

        if os.path.exists(
            path
        ):

            z.write(
                path,
                arcname=filename
            )


# ============================================================
# 25. 自动给出实验判断
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "EXPERIMENT 7 CONCLUSION"
)

print(
    "=" * 70
)


if oof_f1 >= 0.85:

    print(
        "✅ 前景裁剪产生明显提升。"
    )

    print(
        "下一步值得将该专家接入"
        "当前Kaggle 0.88主模型。"
    )


elif oof_f1 > (
    BASELINE_BINARY_F1
    + 0.03
):

    print(
        "✅ 前景裁剪有较明显正收益。"
    )

    print(
        "建议继续优化Foreground Expert，"
        "然后再考虑与主模型融合。"
    )


elif oof_f1 > (
    BASELINE_BINARY_F1
    + 0.01
):

    print(
        "⚠️ 前景裁剪只有轻微提升。"
    )

    print(
        "暂时不足以证明专家系统值得接入主模型。"
    )


else:

    print(
        "❌ 前景裁剪没有带来足够收益。"
    )

    print(
        "建议停止二分类专家路线，"
        "转向全局模型多尺度/高分辨率优化。"
    )


print(
    "\nOOF:",
    OOF_PATH
)

print(
    "Summary:",
    SUMMARY_PATH
)

print(
    "Models:",
    ZIP_PATH
)

print(
    "Preview:",
    PREVIEW_PATH
)

print(
    "\n✅ Experiment 7 finished."
)

In [ ]:
# ============================================================
# Experiment 8
# Global 5-Class EfficientNet-B0
# 320x320 + 5-Fold + OOF
#
# 与原 EfficientNet-B0 224 实验相比：
# 主要变量只有输入分辨率：
# 224x224 -> 320x320
#
# 输出：
# 1. 5个 efficientnet320_fold*.pth
# 2. efficientnet320_oof.csv
# 3. efficientnet320_summary.csv
# 4. efficientnet320_models.zip
# ============================================================

import os
import gc
import random
import zipfile
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. Paths
# ============================================================

DATA_ROOT = str(DATA_DIR)

TRAIN_DIR = os.path.join(
    DATA_ROOT,
    "train"
)

WORK_DIR = str(ARTIFACT_DIR)

os.makedirs(
    WORK_DIR,
    exist_ok=True
)


# ============================================================
# 2. Config
# ============================================================

SEED = 42

N_SPLITS = 5

EPOCHS = 15

BATCH_SIZE = 16

IMAGE_SIZE = 320

RESIZE_SIZE = 352

LR = 1e-4

WEIGHT_DECAY = 1e-4


# 之前 EfficientNet-B0 224 的 5-fold Mean F1
PREVIOUS_224_MEAN_F1 = 0.8998457180254356


classes = [
    "Black-grass",
    "Common wheat",
    "Loose Silky-bent",
    "Scentless Mayweed",
    "Sugar beet"
]

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

idx_to_class = {
    idx: cls
    for cls, idx in class_to_idx.items()
}

NUM_CLASSES = len(classes)


# ============================================================
# 3. Seed
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# 4. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("Device:", device)
print("=" * 70)


# ============================================================
# 5. Build DataFrame
# ============================================================

records = []

for class_name in classes:

    class_dir = os.path.join(
        TRAIN_DIR,
        class_name
    )

    if not os.path.exists(class_dir):

        raise FileNotFoundError(
            class_dir
        )

    # 保持之前数据读取逻辑一致
    for filename in os.listdir(
        class_dir
    ):

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            records.append({
                "path": os.path.join(
                    class_dir,
                    filename
                ),

                "filename": filename,

                "label_name": class_name,

                "label": class_to_idx[
                    class_name
                ]
            })


df = pd.DataFrame(
    records
)


print(
    "\nTotal samples:",
    len(df)
)

print(
    "\nClass distribution:"
)

print(
    df[
        "label_name"
    ].value_counts()
)


if len(df) != 500:

    print(
        "\n⚠️ 预期500张训练图，"
        f"实际找到 {len(df)} 张。"
    )


# ============================================================
# 6. Transform
#
# 对照224版本：
# Resize 256 -> RandomResizedCrop 224
#
# 现在：
# Resize 352 -> RandomResizedCrop 320
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            RESIZE_SIZE,
            RESIZE_SIZE
        )
    ),

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        20
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 7. Dataset
# ============================================================

class PlantDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        image = Image.open(
            row["path"]
        ).convert(
            "RGB"
        )

        label = int(
            row["label"]
        )

        if self.transform:

            image = self.transform(
                image
            )

        return (
            image,
            label
        )


# ============================================================
# 8. Model
# ============================================================

def create_model(
    pretrained=True
):

    weights = (
        models
        .EfficientNet_B0_Weights
        .DEFAULT
        if pretrained
        else None
    )

    model = (
        models
        .efficientnet_b0(
            weights=weights
        )
    )

    in_features = (
        model
        .classifier[1]
        .in_features
    )

    model.classifier[1] = (
        nn.Linear(
            in_features,
            NUM_CLASSES
        )
    )

    return model.to(
        device
    )


# ============================================================
# 9. Evaluation
# ============================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_preds = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item()
                * images.size(0)
            )

            preds = logits.argmax(
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    val_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    val_acc = accuracy_score(
        all_labels,
        all_preds
    )

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return (
        val_loss,
        val_acc,
        val_f1
    )


# ============================================================
# 10. Load checkpoint
# ============================================================

def load_checkpoint(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )

    model.load_state_dict(
        state
    )

    return model


# ============================================================
# 11. Build same 5 folds
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

splits = list(
    skf.split(
        df,
        df["label"]
    )
)


# ============================================================
# 12. Train one fold
# ============================================================

def train_one_fold(
    fold,
    train_idx,
    val_idx
):

    print(
        "\n" + "=" * 70
    )

    print(
        f"EfficientNet-B0 320 | Fold {fold}"
    )

    print(
        "=" * 70
    )


    set_seed(
        SEED + fold
    )


    train_df = (
        df.iloc[
            train_idx
        ]
        .reset_index(
            drop=True
        )
    )

    val_df = (
        df.iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    train_dataset = PlantDataset(
        train_df,
        train_transform
    )

    val_dataset = PlantDataset(
        val_df,
        val_transform
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    model = create_model(
        pretrained=True
    )


    criterion = (
        nn.CrossEntropyLoss()
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )
    )


    best_f1 = -1.0
    best_acc = 0.0
    best_epoch = 0


    checkpoint_path = os.path.join(
        WORK_DIR,
        f"efficientnet320_fold{fold}.pth"
    )


    for epoch in range(
        EPOCHS
    ):

        # -----------------------------
        # Train
        # -----------------------------

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0


        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad()


            logits = model(
                images
            )


            loss = criterion(
                logits,
                labels
            )


            loss.backward()


            optimizer.step()


            running_loss += (
                loss.item()
                * images.size(0)
            )


            preds = logits.argmax(
                dim=1
            )


            correct += (
                preds == labels
            ).sum().item()


            total += (
                labels.size(0)
            )


        train_loss = (
            running_loss
            /
            len(train_dataset)
        )


        train_acc = (
            correct
            /
            total
        )


        # -----------------------------
        # Validation
        # -----------------------------

        (
            val_loss,
            val_acc,
            val_f1
        ) = evaluate(
            model,
            val_loader,
            criterion
        )


        scheduler.step(
            val_f1
        )


        current_lr = (
            optimizer
            .param_groups[0][
                "lr"
            ]
        )


        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:.4f} | "
            f"Val F1 {val_f1:.4f} | "
            f"LR {current_lr:.2e}"
        )


        if val_f1 > best_f1:

            best_f1 = val_f1

            best_acc = val_acc

            best_epoch = (
                epoch + 1
            )


            torch.save(
                model.state_dict(),
                checkpoint_path
            )


            print(
                f"  -> Saved best "
                f"F1={best_f1:.4f}"
            )


    # -----------------------------
    # Cleanup
    # -----------------------------

    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {

        "fold": fold,

        "best_f1": best_f1,

        "best_acc": best_acc,

        "best_epoch": best_epoch,

        "path": checkpoint_path
    }


# ============================================================
# 13. Train all 5 folds
# ============================================================

results = []


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    result = train_one_fold(
        fold,
        train_idx,
        val_idx
    )

    results.append(
        result
    )


# ============================================================
# 14. Fold summary
# ============================================================

fold_scores = [
    x["best_f1"]
    for x in results
]


mean_fold_f1 = np.mean(
    fold_scores
)

std_fold_f1 = np.std(
    fold_scores
)


print(
    "\n\n" + "=" * 70
)

print(
    "EFFICIENTNET-B0 320 5-FOLD RESULTS"
)

print(
    "=" * 70
)


for result in results:

    print(
        f"Fold {result['fold']} | "
        f"F1={result['best_f1']:.4f} | "
        f"Acc={result['best_acc']:.4f} | "
        f"Best Epoch={result['best_epoch']}"
    )


print(
    "\nMean Best-Fold F1:",
    mean_fold_f1
)

print(
    "Std:",
    std_fold_f1
)


print(
    "\nPrevious EfficientNet-B0 224:"
)

print(
    PREVIOUS_224_MEAN_F1
)


fold_delta = (
    mean_fold_f1
    -
    PREVIOUS_224_MEAN_F1
)


print(
    "\n320 vs 224 Delta:"
)

print(
    f"{fold_delta:+.4f}"
)


# ============================================================
# 15. Generate OOF probabilities
# ============================================================

oof_probs = np.zeros(
    (
        len(df),
        NUM_CLASSES
    ),
    dtype=np.float32
)


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    print(
        f"\nGenerating OOF | "
        f"Fold {fold}"
    )


    val_df = (
        df.iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    val_dataset = PlantDataset(
        val_df,
        val_transform
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    model = create_model(
        pretrained=False
    )


    checkpoint_path = os.path.join(
        WORK_DIR,
        f"efficientnet320_fold{fold}.pth"
    )


    model = load_checkpoint(
        model,
        checkpoint_path
    )


    model.eval()


    fold_probs = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )


            logits = model(
                images
            )


            probs = F.softmax(
                logits,
                dim=1
            )


            fold_probs.append(
                probs
                .cpu()
                .numpy()
            )


    fold_probs = np.concatenate(
        fold_probs,
        axis=0
    )


    oof_probs[
        val_idx
    ] = fold_probs


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 16. OOF metrics
# ============================================================

y_true = (
    df["label"]
    .values
)


oof_preds = (
    oof_probs
    .argmax(
        axis=1
    )
)


oof_acc = accuracy_score(
    y_true,
    oof_preds
)


oof_f1 = f1_score(
    y_true,
    oof_preds,
    average="macro"
)


print(
    "\n" + "=" * 70
)

print(
    "EFFICIENTNET-B0 320 OOF"
)

print(
    "=" * 70
)


print(
    f"OOF Accuracy: "
    f"{oof_acc:.4f}"
)

print(
    f"OOF Macro F1: "
    f"{oof_f1:.4f}"
)


# ============================================================
# 17. Per-class report
# ============================================================

print(
    "\nClassification Report:\n"
)


print(
    classification_report(
        y_true,
        oof_preds,
        target_names=classes,
        digits=4
    )
)


# ============================================================
# 18. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    oof_preds
)


print(
    "\nConfusion Matrix:"
)

print(
    cm
)


# ============================================================
# 19. Save OOF
# ============================================================

oof_df = (
    df[
        [
            "filename",
            "path",
            "label_name",
            "label"
        ]
    ]
    .copy()
)


for class_idx, class_name in enumerate(
    classes
):

    safe_name = (
        class_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )

    oof_df[
        f"p_{safe_name}"
    ] = (
        oof_probs[
            :,
            class_idx
        ]
    )


oof_df[
    "prediction"
] = (
    oof_preds
)


oof_df[
    "prediction_name"
] = [
    idx_to_class[
        int(x)
    ]
    for x in oof_preds
]


OOF_PATH = os.path.join(
    WORK_DIR,
    "efficientnet320_oof.csv"
)


oof_df.to_csv(
    OOF_PATH,
    index=False
)


# ============================================================
# 20. Save summary
# ============================================================

summary_df = pd.DataFrame({

    "metric": [
        "previous_224_mean_fold_f1",
        "efficientnet320_mean_fold_f1",
        "fold_delta",
        "efficientnet320_std_fold_f1",
        "efficientnet320_oof_f1",
        "efficientnet320_oof_accuracy"
    ],

    "value": [
        PREVIOUS_224_MEAN_F1,
        mean_fold_f1,
        fold_delta,
        std_fold_f1,
        oof_f1,
        oof_acc
    ]
})


SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "efficientnet320_summary.csv"
)


summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 21. Zip 5 models
# ============================================================

ZIP_PATH = os.path.join(
    WORK_DIR,
    "efficientnet320_models.zip"
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        filename = (
            f"efficientnet320_fold{fold}.pth"
        )

        path = os.path.join(
            WORK_DIR,
            filename
        )

        if os.path.exists(
            path
        ):

            z.write(
                path,
                arcname=filename
            )


# ============================================================
# 22. Automatic conclusion
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "EXPERIMENT 8 CONCLUSION"
)

print(
    "=" * 70
)


if mean_fold_f1 >= 0.91:

    print(
        "✅ 320分辨率带来明显提升。"
    )

    print(
        "下一步应将320模型与现有224 ensemble"
        "进行多尺度概率融合。"
    )


elif mean_fold_f1 >= (
    PREVIOUS_224_MEAN_F1
    + 0.005
):

    print(
        "✅ 320分辨率有正收益。"
    )

    print(
        "即使提升不大，也值得测试224+320"
        "多尺度ensemble。"
    )


elif mean_fold_f1 >= 0.895:

    print(
        "⚠️ 320与224单模型性能接近。"
    )

    print(
        "不要直接放弃：错误模式可能不同，"
        "仍值得测试多尺度融合。"
    )


else:

    print(
        "❌ 320单模型表现明显下降。"
    )

    print(
        "单纯提高分辨率并没有带来收益。"
    )


print(
    "\nOOF:",
    OOF_PATH
)

print(
    "Summary:",
    SUMMARY_PATH
)

print(
    "Models:",
    ZIP_PATH
)


print(
    "\n✅ Experiment 8 finished."
)

In [ ]:
# ============================================================
# EfficientNet-B0 320
# 5-Fold Soft Voting -> Kaggle Submission
# Plain inference, NO TTA
# ============================================================

import os
import gc
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


# ============================================================
# 1. Paths
# ============================================================

DATA_ROOT = str(DATA_DIR)

TEST_DIR = os.path.join(
    DATA_ROOT,
    "test"
)

SUBMISSION_PATH = str(SAMPLE_SUBMISSION)

WORK_DIR = str(ARTIFACT_DIR)

OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "submission_effnet320_5fold.csv"
)


# ============================================================
# 2. Classes
# ============================================================

classes = [
    "Black-grass",
    "Common wheat",
    "Loose Silky-bent",
    "Scentless Mayweed",
    "Sugar beet"
]

idx_to_class = {
    idx: cls
    for idx, cls in enumerate(classes)
}

NUM_CLASSES = 5
IMAGE_SIZE = 320
BATCH_SIZE = 16


# ============================================================
# 3. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 4. Check checkpoints
# ============================================================

for fold in range(1, 6):

    path = os.path.join(
        WORK_DIR,
        f"efficientnet320_fold{fold}.pth"
    )

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"缺少模型：{path}"
        )

print("✅ 5个320模型全部存在")


# ============================================================
# 5. Submission / Test files
# ============================================================

submission = pd.read_csv(
    SUBMISSION_PATH
)

test_filenames = (
    submission["ID"]
    .astype(str)
    .tolist()
)


# 自动递归寻找test图片
test_file_map = {}

for root, dirs, files in os.walk(
    TEST_DIR
):

    for filename in files:

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            test_file_map[
                filename
            ] = os.path.join(
                root,
                filename
            )


missing = [
    x
    for x in test_filenames
    if x not in test_file_map
]

if missing:

    raise FileNotFoundError(
        f"有 {len(missing)} 张测试图片找不到"
    )


print(
    "Test samples:",
    len(test_filenames)
)


# ============================================================
# 6. Transform
#    必须和Experiment 8验证集一致
# ============================================================

test_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 7. Dataset
# ============================================================

class TestDataset(Dataset):

    def __init__(
        self,
        filenames,
        file_map,
        transform
    ):

        self.filenames = filenames
        self.file_map = file_map
        self.transform = transform


    def __len__(self):

        return len(
            self.filenames
        )


    def __getitem__(
        self,
        idx
    ):

        filename = (
            self.filenames[idx]
        )

        image = Image.open(
            self.file_map[
                filename
            ]
        ).convert(
            "RGB"
        )

        image = self.transform(
            image
        )

        return image


test_dataset = TestDataset(
    test_filenames,
    test_file_map,
    test_transform
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 8. Model
# ============================================================

def create_model():

    model = models.efficientnet_b0(
        weights=None
    )

    in_features = (
        model
        .classifier[1]
        .in_features
    )

    model.classifier[1] = (
        nn.Linear(
            in_features,
            NUM_CLASSES
        )
    )

    return model.to(
        device
    )


# ============================================================
# 9. Load checkpoint
# ============================================================

def load_checkpoint(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )

    model.load_state_dict(
        state
    )

    return model


# ============================================================
# 10. 5-Fold predictions
# ============================================================

all_fold_probs = []


for fold in range(
    1,
    6
):

    print(
        f"Predict Fold {fold}/5 ..."
    )

    model = create_model()

    path = os.path.join(
        WORK_DIR,
        f"efficientnet320_fold{fold}.pth"
    )

    model = load_checkpoint(
        model,
        path
    )

    model.eval()

    fold_probs = []


    with torch.no_grad():

        for images in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            probs = F.softmax(
                logits,
                dim=1
            )

            fold_probs.append(
                probs
                .cpu()
                .numpy()
            )


    fold_probs = np.concatenate(
        fold_probs,
        axis=0
    )

    all_fold_probs.append(
        fold_probs
    )


    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 11. Fold probability average
# ============================================================

final_probs = np.mean(
    np.stack(
        all_fold_probs,
        axis=0
    ),
    axis=0
)


print(
    "Probability shape:",
    final_probs.shape
)


# ============================================================
# 12. Prediction
# ============================================================

predictions = (
    final_probs.argmax(
        axis=1
    )
)


predicted_classes = [
    idx_to_class[
        int(x)
    ]
    for x in predictions
]


# ============================================================
# 13. Submission
# ============================================================

final_submission = (
    submission.copy()
)

final_submission[
    "Category"
] = predicted_classes


final_submission.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "\nSubmission:"
)

print(
    final_submission.head()
)


print(
    "\nPrediction distribution:"
)

print(
    final_submission[
        "Category"
    ].value_counts()
)


print(
    "\nSaved:"
)

print(
    OUTPUT_PATH
)


print(
    "\n✅ 可以提交Kaggle"
)

In [ ]:
# ============================================================
# Experiment 9
# EfficientNetV2-S + 384x384
# 5-Fold CV + Layer-wise LR + OOF
#
# 输出：
# 1. efficientnetv2s384_fold1~5.pth
# 2. efficientnetv2s384_oof.csv
# 3. efficientnetv2s384_summary.csv
# 4. efficientnetv2s384_models.zip
# ============================================================

import os
import gc
import random
import zipfile
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. Paths
# ============================================================

DATA_ROOT = str(DATA_DIR)

TRAIN_DIR = os.path.join(
    DATA_ROOT,
    "train"
)

WORK_DIR = str(ARTIFACT_DIR)

os.makedirs(
    WORK_DIR,
    exist_ok=True
)


# ============================================================
# 2. Config
# ============================================================

SEED = 42

N_SPLITS = 5

EPOCHS = 15

# 384 + V2-S显存占用明显更高
BATCH_SIZE = 8

IMAGE_SIZE = 384

RESIZE_SIZE = 416

BACKBONE_LR = 3e-5

HEAD_LR = 3e-4

WEIGHT_DECAY = 1e-4


# 已有最佳320模型
PREVIOUS_320_OOF_F1 = 0.9146


classes = [
    "Black-grass",
    "Common wheat",
    "Loose Silky-bent",
    "Scentless Mayweed",
    "Sugar beet"
]

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

idx_to_class = {
    idx: cls
    for cls, idx in class_to_idx.items()
}

NUM_CLASSES = len(classes)


# ============================================================
# 3. Seed
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(SEED)


# ============================================================
# 4. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)

print(
    "Device:",
    device
)

print("=" * 70)


# ============================================================
# 5. Build DataFrame
# ============================================================

records = []


for class_name in classes:

    class_dir = os.path.join(
        TRAIN_DIR,
        class_name
    )

    if not os.path.exists(
        class_dir
    ):

        raise FileNotFoundError(
            class_dir
        )

    for filename in os.listdir(
        class_dir
    ):

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            records.append({

                "path": os.path.join(
                    class_dir,
                    filename
                ),

                "filename": filename,

                "label_name": class_name,

                "label": class_to_idx[
                    class_name
                ]
            })


df = pd.DataFrame(
    records
)


print(
    "\nSamples:",
    len(df)
)

print(
    "\nClass distribution:"
)

print(
    df[
        "label_name"
    ].value_counts()
)


# ============================================================
# 6. Transform
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            RESIZE_SIZE,
            RESIZE_SIZE
        )
    ),

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.82, 1.0)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        20
    ),

    transforms.ColorJitter(
        brightness=0.12,
        contrast=0.12,
        saturation=0.12
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 7. Dataset
# ============================================================

class PlantDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        image = Image.open(
            row["path"]
        ).convert(
            "RGB"
        )

        label = int(
            row["label"]
        )

        if self.transform:

            image = self.transform(
                image
            )

        return (
            image,
            label
        )


# ============================================================
# 8. Model
# ============================================================

def create_model(
    pretrained=True
):

    weights = (
        models
        .EfficientNet_V2_S_Weights
        .DEFAULT
        if pretrained
        else None
    )

    model = (
        models
        .efficientnet_v2_s(
            weights=weights
        )
    )

    in_features = (
        model
        .classifier[1]
        .in_features
    )

    model.classifier[1] = (
        nn.Linear(
            in_features,
            NUM_CLASSES
        )
    )

    return model.to(
        device
    )


# ============================================================
# 9. Evaluation
# ============================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_preds = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item()
                * images.size(0)
            )

            preds = logits.argmax(
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    val_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    val_acc = accuracy_score(
        all_labels,
        all_preds
    )

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return (
        val_loss,
        val_acc,
        val_f1
    )


# ============================================================
# 10. Checkpoint loader
# ============================================================

def load_checkpoint(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )


    model.load_state_dict(
        state
    )

    return model


# ============================================================
# 11. 5-Fold split
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


splits = list(
    skf.split(
        df,
        df["label"]
    )
)


# ============================================================
# 12. Train one fold
# ============================================================

def train_one_fold(
    fold,
    train_idx,
    val_idx
):

    print(
        "\n" + "=" * 70
    )

    print(
        f"EfficientNetV2-S 384 | Fold {fold}"
    )

    print(
        "=" * 70
    )


    set_seed(
        SEED + fold
    )


    train_df = (
        df.iloc[
            train_idx
        ]
        .reset_index(
            drop=True
        )
    )

    val_df = (
        df.iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    train_dataset = PlantDataset(
        train_df,
        train_transform
    )

    val_dataset = PlantDataset(
        val_df,
        val_transform
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    model = create_model(
        pretrained=True
    )


    criterion = (
        nn.CrossEntropyLoss()
    )


    # ========================================================
    # 分层学习率
    #
    # Backbone：小LR
    # Classifier：大LR
    # ========================================================

    optimizer = torch.optim.AdamW(
        [
            {
                "params":
                model.features.parameters(),

                "lr":
                BACKBONE_LR
            },

            {
                "params":
                model.classifier.parameters(),

                "lr":
                HEAD_LR
            }
        ],

        weight_decay=
        WEIGHT_DECAY
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )
    )


    best_f1 = -1.0
    best_acc = 0.0
    best_epoch = 0


    checkpoint_path = os.path.join(
        WORK_DIR,
        f"efficientnetv2s384_fold{fold}.pth"
    )


    for epoch in range(
        EPOCHS
    ):

        model.train()

        running_loss = 0.0
        train_correct = 0
        train_total = 0


        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad()


            logits = model(
                images
            )


            loss = criterion(
                logits,
                labels
            )


            loss.backward()


            optimizer.step()


            running_loss += (
                loss.item()
                * images.size(0)
            )


            preds = logits.argmax(
                dim=1
            )


            train_correct += (
                preds == labels
            ).sum().item()


            train_total += (
                labels.size(0)
            )


        train_loss = (
            running_loss
            /
            len(train_dataset)
        )


        train_acc = (
            train_correct
            /
            train_total
        )


        (
            val_loss,
            val_acc,
            val_f1
        ) = evaluate(
            model,
            val_loader,
            criterion
        )


        scheduler.step(
            val_f1
        )


        backbone_lr = (
            optimizer
            .param_groups[0]["lr"]
        )


        head_lr = (
            optimizer
            .param_groups[1]["lr"]
        )


        print(
            f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:.4f} | "
            f"Val F1 {val_f1:.4f} | "
            f"LR {backbone_lr:.1e}/{head_lr:.1e}"
        )


        if val_f1 > best_f1:

            best_f1 = val_f1

            best_acc = val_acc

            best_epoch = (
                epoch + 1
            )


            torch.save(
                model.state_dict(),
                checkpoint_path
            )


            print(
                f"  -> Saved best "
                f"F1={best_f1:.4f}"
            )


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {

        "fold": fold,

        "best_f1": best_f1,

        "best_acc": best_acc,

        "best_epoch": best_epoch,

        "path": checkpoint_path
    }


# ============================================================
# 13. Train all folds
# ============================================================

results = []


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    result = train_one_fold(
        fold,
        train_idx,
        val_idx
    )

    results.append(
        result
    )


# ============================================================
# 14. Fold summary
# ============================================================

fold_scores = [
    x["best_f1"]
    for x in results
]


mean_fold_f1 = np.mean(
    fold_scores
)


std_fold_f1 = np.std(
    fold_scores
)


print(
    "\n\n" + "=" * 70
)

print(
    "EFFICIENTNETV2-S 384 5-FOLD RESULTS"
)

print(
    "=" * 70
)


for result in results:

    print(
        f"Fold {result['fold']} | "
        f"F1={result['best_f1']:.4f} | "
        f"Acc={result['best_acc']:.4f} | "
        f"Epoch={result['best_epoch']}"
    )


print(
    "\nMean Best-Fold F1:",
    mean_fold_f1
)

print(
    "Std:",
    std_fold_f1
)


# ============================================================
# 15. Generate OOF
# ============================================================

oof_probs = np.zeros(
    (
        len(df),
        NUM_CLASSES
    ),
    dtype=np.float32
)


for fold, (
    train_idx,
    val_idx
) in enumerate(
    splits,
    start=1
):

    print(
        f"\nGenerating OOF | Fold {fold}"
    )


    val_df = (
        df.iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )


    val_dataset = PlantDataset(
        val_df,
        val_transform
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    model = create_model(
        pretrained=False
    )


    checkpoint_path = os.path.join(
        WORK_DIR,
        f"efficientnetv2s384_fold{fold}.pth"
    )


    model = load_checkpoint(
        model,
        checkpoint_path
    )


    model.eval()


    fold_probs = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            probs = F.softmax(
                logits,
                dim=1
            )

            fold_probs.append(
                probs
                .cpu()
                .numpy()
            )


    fold_probs = np.concatenate(
        fold_probs,
        axis=0
    )


    oof_probs[
        val_idx
    ] = fold_probs


    del model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 16. OOF metrics
# ============================================================

y_true = (
    df["label"]
    .values
)


oof_preds = (
    oof_probs
    .argmax(
        axis=1
    )
)


oof_acc = accuracy_score(
    y_true,
    oof_preds
)


oof_f1 = f1_score(
    y_true,
    oof_preds,
    average="macro"
)


print(
    "\n" + "=" * 70
)

print(
    "EFFICIENTNETV2-S 384 OOF"
)

print(
    "=" * 70
)


print(
    f"OOF Accuracy: "
    f"{oof_acc:.4f}"
)

print(
    f"OOF Macro F1: "
    f"{oof_f1:.4f}"
)


print(
    "\nPrevious B0-320 OOF:"
)

print(
    f"{PREVIOUS_320_OOF_F1:.4f}"
)


delta = (
    oof_f1
    -
    PREVIOUS_320_OOF_F1
)


print(
    "\nDelta:"
)

print(
    f"{delta:+.4f}"
)


# ============================================================
# 17. Classification report
# ============================================================

print(
    "\nClassification Report:\n"
)


print(
    classification_report(
        y_true,
        oof_preds,
        target_names=classes,
        digits=4
    )
)


# ============================================================
# 18. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    oof_preds
)


print(
    "\nConfusion Matrix:"
)

print(
    cm
)


# ============================================================
# 19. Save OOF
# ============================================================

oof_df = (
    df[
        [
            "filename",
            "path",
            "label_name",
            "label"
        ]
    ]
    .copy()
)


for class_idx, class_name in enumerate(
    classes
):

    safe_name = (
        class_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )

    oof_df[
        f"p_{safe_name}"
    ] = (
        oof_probs[
            :,
            class_idx
        ]
    )


oof_df[
    "prediction"
] = (
    oof_preds
)


oof_df[
    "prediction_name"
] = [
    idx_to_class[
        int(x)
    ]
    for x in oof_preds
]


OOF_PATH = os.path.join(
    WORK_DIR,
    "efficientnetv2s384_oof.csv"
)


oof_df.to_csv(
    OOF_PATH,
    index=False
)


# ============================================================
# 20. Save summary
# ============================================================

summary_df = pd.DataFrame({

    "metric": [
        "mean_best_fold_f1",
        "std_best_fold_f1",
        "oof_f1",
        "oof_accuracy",
        "previous_b0_320_oof_f1",
        "delta_vs_b0_320"
    ],

    "value": [
        mean_fold_f1,
        std_fold_f1,
        oof_f1,
        oof_acc,
        PREVIOUS_320_OOF_F1,
        delta
    ]
})


SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "efficientnetv2s384_summary.csv"
)


summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 21. Zip models
# ============================================================

ZIP_PATH = os.path.join(
    WORK_DIR,
    "efficientnetv2s384_models.zip"
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        filename = (
            f"efficientnetv2s384_fold{fold}.pth"
        )

        path = os.path.join(
            WORK_DIR,
            filename
        )

        if os.path.exists(
            path
        ):

            z.write(
                path,
                arcname=filename
            )


# ============================================================
# 22. Automatic conclusion
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "EXPERIMENT 9 CONCLUSION"
)

print(
    "=" * 70
)


if oof_f1 >= 0.94:

    print(
        "✅ 非常成功。"
    )

    print(
        "下一步值得训练不同结构模型，"
        "再进行OOF ensemble。"
    )


elif oof_f1 >= 0.925:

    print(
        "✅ 有明显价值。"
    )

    print(
        "建议进行与B0-320的OOF融合。"
    )


elif oof_f1 >= 0.91:

    print(
        "⚠️ 总体性能接近B0-320。"
    )

    print(
        "重点检查Black-grass / Loose Silky-bent"
        "以及错误样本是否互补。"
    )


else:

    print(
        "❌ V2-S 384没有获得预期收益。"
    )

    print(
        "不建议继续扩大同类模型。"
    )


print(
    "\nOOF:",
    OOF_PATH
)

print(
    "Summary:",
    SUMMARY_PATH
)

print(
    "Models:",
    ZIP_PATH
)

print(
    "\n✅ Experiment 9 finished."
)

In [ ]:
# ============================================================
# EfficientNetV2-S 384
# 5-Fold Plain Soft Voting
# Kaggle Submission
# ============================================================

import os
import gc
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


# ============================================================
# 1. Paths
# ============================================================

DATA_ROOT = str(DATA_DIR)

TEST_DIR = os.path.join(
    DATA_ROOT,
    "test"
)

SUBMISSION_PATH = str(SAMPLE_SUBMISSION)

WORK_DIR = str(ARTIFACT_DIR)

OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "submission_v2s384_5fold.csv"
)

PROBS_PATH = os.path.join(
    WORK_DIR,
    "v2s384_test_probs.npy"
)


# ============================================================
# 2. Config
# ============================================================

classes = [
    "Black-grass",
    "Common wheat",
    "Loose Silky-bent",
    "Scentless Mayweed",
    "Sugar beet"
]

idx_to_class = {
    i: c
    for i, c in enumerate(classes)
}

NUM_CLASSES = 5

IMAGE_SIZE = 384

BATCH_SIZE = 8


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 3. Check models
# ============================================================

for fold in range(1, 6):

    path = os.path.join(
        WORK_DIR,
        f"efficientnetv2s384_fold{fold}.pth"
    )

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"缺少模型：{path}"
        )

print("✅ 5个 EfficientNetV2-S 384 模型全部存在")


# ============================================================
# 4. Submission template
# ============================================================

submission = pd.read_csv(
    SUBMISSION_PATH
)

test_filenames = (
    submission["ID"]
    .astype(str)
    .tolist()
)


# ============================================================
# 5. Find test images
# ============================================================

test_file_map = {}

for root, dirs, files in os.walk(
    TEST_DIR
):

    for filename in files:

        if filename.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg",
                ".bmp",
                ".webp"
            )
        ):

            test_file_map[
                filename
            ] = os.path.join(
                root,
                filename
            )


missing = [
    f
    for f in test_filenames
    if f not in test_file_map
]

if missing:

    raise FileNotFoundError(
        f"找不到 {len(missing)} 张测试图片"
    )

print(
    "Test samples:",
    len(test_filenames)
)


# ============================================================
# 6. Transform
#    必须和384验证阶段一致
# ============================================================

test_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 7. Dataset
# ============================================================

class TestDataset(Dataset):

    def __init__(
        self,
        filenames,
        file_map,
        transform
    ):

        self.filenames = filenames
        self.file_map = file_map
        self.transform = transform


    def __len__(self):

        return len(
            self.filenames
        )


    def __getitem__(
        self,
        idx
    ):

        filename = (
            self.filenames[idx]
        )

        image = Image.open(
            self.file_map[
                filename
            ]
        ).convert(
            "RGB"
        )

        image = self.transform(
            image
        )

        return image


test_dataset = TestDataset(
    test_filenames,
    test_file_map,
    test_transform
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 8. Model
# ============================================================

def create_model():

    model = models.efficientnet_v2_s(
        weights=None
    )

    in_features = (
        model
        .classifier[1]
        .in_features
    )

    model.classifier[1] = (
        nn.Linear(
            in_features,
            NUM_CLASSES
        )
    )

    return model.to(
        device
    )


# ============================================================
# 9. Load
# ============================================================

def load_model(
    model,
    path
):

    try:

        state = torch.load(
            path,
            map_location=device,
            weights_only=True
        )

    except TypeError:

        state = torch.load(
            path,
            map_location=device
        )

    model.load_state_dict(
        state
    )

    return model


# ============================================================
# 10. Predict all folds
# ============================================================

all_fold_probs = []


for fold in range(1, 6):

    print(
        f"Predicting Fold {fold}/5..."
    )

    model = create_model()

    path = os.path.join(
        WORK_DIR,
        f"efficientnetv2s384_fold{fold}.pth"
    )

    model = load_model(
        model,
        path
    )

    model.eval()

    probs_list = []


    with torch.no_grad():

        for images in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            probs = F.softmax(
                logits,
                dim=1
            )

            probs_list.append(
                probs
                .cpu()
                .numpy()
            )


    fold_probs = np.concatenate(
        probs_list,
        axis=0
    )

    all_fold_probs.append(
        fold_probs
    )


    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 11. 5-fold Soft Voting
# ============================================================

final_probs = np.mean(
    np.stack(
        all_fold_probs,
        axis=0
    ),
    axis=0
)


print(
    "Probability shape:",
    final_probs.shape
)


# 顺便保存概率
np.save(
    PROBS_PATH,
    final_probs
)


# ============================================================
# 12. Prediction
# ============================================================

pred_ids = final_probs.argmax(
    axis=1
)


pred_classes = [
    idx_to_class[
        int(x)
    ]
    for x in pred_ids
]


# ============================================================
# 13. Submission
# ============================================================

final_submission = (
    submission.copy()
)

final_submission[
    "Category"
] = pred_classes


final_submission.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "\nPrediction distribution:"
)

print(
    final_submission[
        "Category"
    ].value_counts()
)


print(
    "\nSubmission saved:"
)

print(
    OUTPUT_PATH
)

print(
    "\nProbabilities saved:"
)

print(
    PROBS_PATH
)

print(
    "\n✅ 可以提交 Kaggle"
)